<a href="https://colab.research.google.com/github/ArrenOnom/eritrea-cmip6-temperature-evaluation/blob/main/ERITREA_CMIP6_TEMPERATURE_EVALUATION_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Repository Initialization & Project Directory Architecture

### 0.1 Environment Setup & Reproducible Path Registry
This initialization cell clones the `eritrea-cmip6-temperature-evaluation` repository into Google Colab, sets `/content/eritrea-cmip6-temperature-evaluation` as the active working directory, ensures `4_NOTEBOOKS` and `5_OUTPUTS` exist, and registers standardized `pathlib.Path` constants for all downstream evaluation, ensemble, trend, and spatial mapping scripts.

### 0.2 Directory Structure
* **Data Layers (`1_DATA`):** Raw historical (`1A_HISTORICAL_RAW`), local station (`1B_LOCAL_STATION_RAW`), and top-performing model (`1C_TOP_PERFORMING_MODEL_RAW`) tables, alongside historical (`1D`–`1F`) and top-performing model (`1G`–`1I`) GeoTIFF rasters.
* **Scripts, Vectors & Outputs (`2_GEE_SCRIPTS` – `5_OUTPUTS`):** Google Earth Engine scripts, study area agro-ecological zone shapefiles (`3_SHAPEFILE_OF_STUDY_AREA`), notebooks, and exported tables/figures.

In [ ]:
# ==============================================================================
# SCRIPT 0.0: REPOSITORY INITIALIZATION & PROJECT DIRECTORY ARCHITECTURE
# Project    : Eritrea CMIP6 Temperature Evaluation (1981-2014 & 2026-2100)
# Repository : https://github.com/ArrenOnom/eritrea-cmip6-temperature-evaluation
# Purpose    : Clone repo into Colab, set working directory, ensure notebook &
#              output directories exist, and expose standardized Path objects.
# ==============================================================================

import os
from pathlib import Path

# ------------------------------------------------------------------------------
# STEP 1: Clone Repository (If Not Already Present) & Set Working Directory
# ------------------------------------------------------------------------------
REPO_URL = (
    "https://github.com/ArrenOnom/eritrea-cmip6-temperature-evaluation.git"
)
ROOT = Path("/content/eritrea-cmip6-temperature-evaluation")

print("=" * 88)
print("SCRIPT 1.0: ERITREA CMIP6 TEMPERATURE EVALUATION — WORKSPACE INITIALIZATION")
print("=" * 88)
print(f"[STEP 1] Target Root : {ROOT}")

if not ROOT.exists():
  print("         Cloning repository from GitHub...")
  os.system(f"git clone {REPO_URL} {ROOT}")
else:
  print("         Repository directory already exists; skipping clone.")

if ROOT.exists():
  os.chdir(ROOT)
  print(f"         Active CWD  : {Path.cwd()}")

# ------------------------------------------------------------------------------
# STEP 2: Define Standardized Project Paths & Ensure Output/Notebook Folders
# ------------------------------------------------------------------------------
DATA = ROOT / "1_DATA"

# Raw Tabular & Climate Datasets
HISTORICAL_RAW = DATA / "1A_HISTORICAL_RAW"
LOCAL_STATION_RAW = DATA / "1B_LOCAL_STATION_RAW"
TOP_MODEL_RAW = DATA / "1C_TOP_PERFORMING_MODEL_RAW"

# Historical Baseline GeoTIFF Directories
HIST_MIN = DATA / "1D_HISTORICAL_MIN_TEMP_GEOTIFF"
HIST_MEAN = DATA / "1E_HISTORICAL_MEAN_TEMP_GEOTIFF"
HIST_MAX = DATA / "1F_HISTORICAL_MAX_TEMP_GEOTIFF"

# Top-Performing Model GeoTIFF Directories
TOP_MIN = DATA / "1G_TOP_PERFORMING_MIN_TEMP_GEOTIFF"
TOP_MEAN = DATA / "1H_TOP_PERFORMING_MEAN_TEMP_GEOTIFF"
TOP_MAX = DATA / "1I_TOP_PERFORMING_MAX_TEMP_GEOTIFF"

# Google Earth Engine Scripts, Vector Boundaries, Notebooks & Outputs
GEE_SCRIPTS = ROOT / "2_GEE_SCRIPTS"
SHAPEFILES = ROOT / "3_SHAPEFILE_OF_STUDY_AREA"
NOTEBOOKS = ROOT / "4_NOTEBOOKS"
OUTPUTS = ROOT / "5_OUTPUTS"

NOTEBOOKS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# STEP 3: Audit Top-Level & Data Subdirectories
# ------------------------------------------------------------------------------
print("\n[STEP 3] Auditing project directory structure...")
print("-" * 88)

top_dirs = [p for p in sorted(ROOT.iterdir()) if p.is_dir()]
for d in top_dirs:
  n_files = sum(1 for _ in d.rglob("*") if _.is_file())
  print(f"  • {d.name:<32} ({n_files} files)")

if DATA.exists():
  print("\n  Data Subdirectories (1_DATA):")
  for sub in sorted(DATA.iterdir()):
    if sub.is_dir():
      n_sub_files = sum(1 for _ in sub.rglob("*") if _.is_file())
      print(f"    - {sub.name:<36} ({n_sub_files} files)")

print("\n" + "=" * 88)
print("[COMPLETE] Workspace initialized and path variables ready for downstream scripts.")
print("=" * 88)

# 1. Point-to-Pixel Validation of ERA5 Reanalysis Temperature Data Across Eritrea

## 1.1 Objective & Evaluation Framework
This notebook performs a point-to-pixel statistical validation of **ERA5 reanalysis** monthly air temperature products—Maximum ($T_{\max}$), Minimum ($T_{\min}$), and Mean ($T_{\text{mean}}$)—against ground-based meteorological station observations across **12 stations in Eritrea**. Each station is evaluated independently across its complete available observation record ($n$) to avoid truncating long-term station series to a narrow common window.

## 1.2 Statistical Performance Metrics
Five complementary statistical metrics are computed for each station and temperature variable (where $O_i$ denotes station observations, $E_i$ denotes ERA5 grid estimates, and $n$ is the number of valid paired months):

1. **Mean Bias Error ($\text{Bias}$, $^\circ\text{C}$):** Quantifies systematic warm ($+$) or cold ($-$) offsets:
   $$\text{Bias} = \frac{1}{n}\sum_{i=1}^{n}(E_i - O_i)$$

2. **Mean Absolute Error ($\text{MAE}$, $^\circ\text{C}$):** Measures average error magnitude in native temperature units without sign cancellation:
   $$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|E_i - O_i|$$

3. **Root Mean Square Error ($\text{RMSE}$, $^\circ\text{C}$):** Quantifies overall error magnitude while penalizing larger temperature deviations:
   $$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(E_i - O_i)^2}$$

4. **Coefficient of Determination ($R^2$):** Evaluates linear correlation and temporal/seasonal phase agreement ($0 \le R^2 \le 1$):
   $$R^2 = \left(\frac{\sum_{i=1}^{n}(O_i - \bar{O})(E_i - \bar{E})}{\sqrt{\sum_{i=1}^{n}(O_i - \bar{O})^2 \sum_{i=1}^{n}(E_i - \bar{E})^2}}\right)^2$$

5. **Kling-Gupta Efficiency ($\text{KGE}$):** Integrates correlation ($r$), variability ratio ($\alpha = \sigma_E / \sigma_O$), and mean bias ratio ($\beta = \mu_E / \mu_O$) into a single skill score ($-\infty < \text{KGE} \le 1$):
   $$\text{KGE} = 1 - \sqrt{(r - 1)^2 + (\alpha - 1)^2 + (\beta - 1)^2}$$

## 1.3 Validation Grading Criteria
Following established climatological and hydrological evaluation benchmarks (Gupta et al., 2009; Moriasi et al., 2015), each station–variable pair is assigned one of three performance verdicts:
* **Pass:** $\text{KGE} \ge 0.70$, $R^2 \ge 0.70$, and $|\text{Bias}| \le 2.5^\circ\text{C}$ (strong temporal, amplitude, and magnitude agreement).
* **Pass w/ Bias Correction:** $\text{KGE} \ge 0.50$ or $R^2 \ge 0.50$ (captures seasonal dynamics reliably, but exhibits systematic elevation- or terrain-induced thermal offset requiring bias correction).
* **Fail:** $\text{KGE} < 0.50$ and $R^2 < 0.50$ (insufficient seasonal/temporal correspondence).

In [ ]:
# ==============================================================================
# SCRIPT 1.A: POINT-TO-PIXEL VALIDATION OF ERA5 TEMPERATURE DATA (ERITREA)
# Variables Evaluated : TMAX, TMIN, TMEAN (Monthly, °C)
# Statistical Suite   : Bias (MBE), MAE, RMSE, R², and KGE
# ==============================================================================

import glob
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Input and Output Directory Paths
# ------------------------------------------------------------------------------
INPUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/1_DATA/ERA5_STATION_VALIDATION_FILES"
OUTPUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/1_DATA/ERA5_VALIDATION_OUTPUTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

station_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
if not station_files:
  raise FileNotFoundError(f"No station CSV files found in: {INPUT_DIR}")

print(f"[STEP 1] Loaded {len(station_files)} station files from:\n  {INPUT_DIR}")


# ------------------------------------------------------------------------------
# STEP 2: Define Validation Verdict / Grading Function
# ------------------------------------------------------------------------------
def assign_validation_grade(r2: float, kge: float, bias: float) -> str:
  """Assigns a standardized performance verdict based on R², KGE, and Bias."""
  if kge >= 0.70 and r2 >= 0.70:
    if abs(bias) > 2.5:
      return "Pass w/ Bias Correction (High R²/KGE, systematic offset)"
    return "Pass"
  elif kge >= 0.50 or r2 >= 0.50:
    return "Pass w/ Bias Correction"
  else:
    return "Fail (Poor R²/KGE)"


# ------------------------------------------------------------------------------
# STEP 3: Define Multi-Metric Statistical Evaluation Function
# ------------------------------------------------------------------------------
def calculate_temperature_metrics(obs: pd.Series, sim: pd.Series) -> dict:
  """Computes Bias, MAE, RMSE, R², and KGE on pairwise complete observations."""
  paired = pd.DataFrame({"obs": obs, "sim": sim}).dropna()
  o = paired["obs"].to_numpy(dtype=float)
  s = paired["sim"].to_numpy(dtype=float)
  n = len(o)

  # Error magnitude and systematic offset (°C)
  diff = s - o
  bias = float(np.mean(diff))
  mae = float(np.mean(np.abs(diff)))
  rmse = float(np.sqrt(np.mean(diff**2)))

  # Coefficient of Determination (R²)
  r = float(np.corrcoef(o, s)[0, 1])
  r2 = float(r**2)

  # Kling-Gupta Efficiency (KGE) components
  alpha = float(np.std(s, ddof=1) / np.std(o, ddof=1))  # Variability ratio
  beta = float(np.mean(s) / np.mean(o))  # Mean bias ratio
  kge = float(
      1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)
  )

  return {
      "n": n,
      "Bias_C": round(bias, 3),
      "MAE_C": round(mae, 3),
      "RMSE_C": round(rmse, 3),
      "R2": round(r2, 3),
      "KGE": round(kge, 3),
      "Final_Grade": assign_validation_grade(r2, kge, bias),
  }


# ------------------------------------------------------------------------------
# STEP 4: Execute Per-Station Validation Across TMAX, TMIN, and TMEAN
# ------------------------------------------------------------------------------
TEMPERATURE_VARS = [
    ("TMAX", "TMAX", "ERA5_TMAX"),
    ("TMIN", "TMIN", "ERA5_TMIN"),
    ("TMEAN", "TMEAN", "ERA5_TMEAN"),
]

station_results = []
for file_path in station_files:
  station_name = (
      os.path.basename(file_path)
      .replace("CLEAN_", "")
      .replace("_4_VALIDATION.csv", "")
      .replace(".csv", "")
  )
  df = pd.read_csv(file_path)

  lat = round(float(df["LAT"].iloc[0]), 4)
  lon = round(float(df["LON"].iloc[0]), 4)
  period = (
      f"{int(df['YEAR'].iloc[0])}-{int(df['MONTH'].iloc[0]):02d} to "
      f"{int(df['YEAR'].iloc[-1])}-{int(df['MONTH'].iloc[-1]):02d}"
  )

  for var_label, obs_col, era_col in TEMPERATURE_VARS:
    metrics = calculate_temperature_metrics(df[obs_col], df[era_col])
    station_results.append({
        "Station": station_name,
        "Lat": lat,
        "Lon": lon,
        "Period": period,
        "Variable": var_label,
        **metrics,
    })

results_df = pd.DataFrame(station_results)

# ------------------------------------------------------------------------------
# STEP 5: Aggregate Network-Wide Summary Statistics by Variable
# ------------------------------------------------------------------------------
summary_df = (
    results_df.groupby("Variable", sort=False)
    .agg(
        Total_Stations=("Station", "count"),
        Mean_Bias_C=("Bias_C", "mean"),
        Mean_MAE_C=("MAE_C", "mean"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Mean_R2=("R2", "mean"),
        Mean_KGE=("KGE", "mean"),
        Acceptable_Stations=(
            "Final_Grade",
            lambda s: int(s.str.startswith("Pass").sum()),
        ),
    )
    .reset_index()
    .round(3)
)

# ------------------------------------------------------------------------------
# STEP 6: Export Tables to CSV and Display Formatted Outputs
# ------------------------------------------------------------------------------
station_csv_path = os.path.join(
    OUTPUT_DIR, "1_ERA5_STATIONWISE_VALIDATION_RESULTS.csv"
)
summary_csv_path = os.path.join(
    OUTPUT_DIR, "2_ERA5_NETWORK_SUMMARY_BY_VARIABLE.csv"
)

results_df.to_csv(station_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)

print(
    "\n=============================================================================="
)
print("TABLE 1.1: STATION-WISE ERA5 TEMPERATURE VALIDATION METRICS & VERDICT")
print(
    "=============================================================================="
)
print(results_df.to_string(index=False))

print(
    "\n=============================================================================="
)
print("TABLE 1.2: NETWORK-WIDE SUMMARY ACROSS ALL 12 ERITREAN STATIONS")
print(
    "=============================================================================="
)
print(summary_df.to_string(index=False))

print("\n[STEP 6] Output files successfully exported:")
print(f"  -> Station-wise table : {station_csv_path}")
print(f"  -> Summary table      : {summary_csv_path}")

In [ ]:
# ==============================================================================
# SCRIPT 1.B: PUBLICATION FIGURE 2 - STATION-WISE BIAS & KGE COMPARISON
# Input  : 1_ERA5_STATIONWISE_VALIDATION_RESULTS.csv
# Output : Figure2_ERA5_Stationwise_Bias_KGE.png & .tif (680 DPI)
# ==============================================================================

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Paths & Strict Journal Styling Configuration
# ------------------------------------------------------------------------------
OUTPUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/1_DATA/ERA5_VALIDATION_OUTPUTS"
INPUT_CSV = os.path.join(OUTPUT_DIR, "1_ERA5_STATIONWISE_VALIDATION_RESULTS.csv")

PNG_OUT = os.path.join(OUTPUT_DIR, "Figure2_ERA5_Stationwise_Bias_KGE.png")
TIF_OUT = os.path.join(OUTPUT_DIR, "Figure2_ERA5_Stationwise_Bias_KGE.tif")

# Global Matplotlib typography and canvas rules
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

# ------------------------------------------------------------------------------
# STEP 2: Load & Pivot Validation Results
# ------------------------------------------------------------------------------
df = pd.read_csv(INPUT_CSV)

stations = list(dict.fromkeys(df["Station"].tolist()))
var_order = ["TMAX", "TMIN", "TMEAN"]
var_labels = {
    "TMAX": "Tmax (°C)",
    "TMIN": "Tmin (°C)",
    "TMEAN": "Tmean (°C)",
}

# Colorblind-safe high-contrast palette for the 3 temperature variables
var_colors = {
    "TMAX": "#D95F02",  # Strong burnt orange
    "TMIN": "#1B7837",  # Deep forest green
    "TMEAN": "#2166AC",  # Strong cobalt blue
}


def pivot_metric(metric_col: str) -> dict:
  piv = df.pivot(index="Station", columns="Variable", values=metric_col)
  piv = piv.reindex(index=stations, columns=var_order)
  return {v: piv[v].to_numpy(dtype=float) for v in var_order}


bias_data = pivot_metric("Bias_C")
kge_data = pivot_metric("KGE")


# ------------------------------------------------------------------------------
# STEP 3: Helper Function to Harden Spines, Ticks, Grid & Inline Labels
# ------------------------------------------------------------------------------
def style_axis(ax: plt.Axes, ylabel: str, panel_tag: str) -> None:
  """Applies strict publication spines, heavy ticks, white canvas, and dashed grid."""
  ax.set_facecolor("#ffffff")
  ax.set_axisbelow(True)

  # All four plot spines solid black with heavy line thickness (>= 3.5)
  for spine_name in ["top", "bottom", "left", "right"]:
    spine = ax.spines[spine_name]
    spine.set_visible(True)
    spine.set_color("#000000")
    spine.set_linewidth(3.5)

  # Thick, dark ticks with length=7 and width=3.5
  ax.tick_params(
      axis="both",
      which="major",
      direction="out",
      length=7,
      width=3.5,
      colors="#000000",
      labelsize=12,
  )
  for tick_lbl in ax.get_xticklabels() + ax.get_yticklabels():
    tick_lbl.set_fontweight("bold")
    tick_lbl.set_color("#000000")
    tick_lbl.set_fontsize(12)

  # Clean light-grey dashed grid behind data vectors
  ax.grid(
      axis="y",
      which="major",
      linestyle="--",
      linewidth=1.5,
      color="#cccccc",
      zorder=0,
  )

  # Bold 14pt axis label and bold panel tag
  ax.set_ylabel(
      ylabel, fontsize=14, fontweight="bold", color="#000000", labelpad=10
  )
  ax.text(
      0.012,
      0.96,
      panel_tag,
      transform=ax.transAxes,
      fontsize=14,
      fontweight="bold",
      color="#000000",
      ha="left",
      va="top",
      bbox=dict(
          boxstyle="square,pad=0.25",
          facecolor="#ffffff",
          edgecolor="#000000",
          linewidth=2.5,
      ),
      zorder=6,
  )


# ------------------------------------------------------------------------------
# STEP 4: Construct Two-Panel Grouped Bar Figure (Bias & KGE)
# ------------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(18, 12), sharex=True, dpi=680
)
fig.patch.set_facecolor("#ffffff")

x = np.arange(len(stations))
bar_width = 0.25
offsets = [-bar_width, 0.0, bar_width]

# --- Panel A: Mean Bias (°C) ---
style_axis(ax1, "Mean Bias (°C)", "(a) Station-Wise Bias (ERA5 − Station)")

# Heavy zero-bias reference line (linewidth >= 4.5)
ax1.axhline(
    0.0,
    color="#000000",
    linestyle="-",
    linewidth=4.5,
    zorder=2,
    label="Zero Bias (0.0 °C)",
)

for idx, var in enumerate(var_order):
  vals = bias_data[var]
  pos_x = x + offsets[idx]
  bars = ax1.bar(
      pos_x,
      vals,
      width=bar_width,
      color=var_colors[var],
      edgecolor="#000000",
      linewidth=3.0,
      label=var_labels[var],
      zorder=3,
  )

  # Inline value annotations in bold black
  for rect, val in zip(bars, vals):
    if val >= 0:
      y_pos = val + 0.25
      va_align = "bottom"
    else:
      y_pos = val - 0.25
      va_align = "top"
    ax1.text(
        rect.get_x() + rect.get_width() / 2.0,
        y_pos,
        f"{val:+.1f}",
        ha="center",
        va=va_align,
        fontsize=9.5,
        fontweight="bold",
        color="#000000",
        rotation=90,
        zorder=5,
    )

ax1.set_ylim(-8.2, 8.8)

# --- Panel B: Kling-Gupta Efficiency (KGE) ---
style_axis(ax2, "KGE Score", "(b) Station-Wise KGE Skill Score")

# Heavy skill threshold lines (linewidth >= 4.5)
ax2.axhline(
    0.70,
    color="#111111",
    linestyle="-",
    linewidth=4.5,
    zorder=2,
    label="Pass Threshold (KGE = 0.70)",
)
ax2.axhline(
    0.50,
    color="#444444",
    linestyle="-.",
    linewidth=4.5,
    zorder=2,
    label="Bias-Correction Threshold (KGE = 0.50)",
)

for idx, var in enumerate(var_order):
  vals = kge_data[var]
  pos_x = x + offsets[idx]
  bars = ax2.bar(
      pos_x,
      vals,
      width=bar_width,
      color=var_colors[var],
      edgecolor="#000000",
      linewidth=3.0,
      zorder=3,
  )

  # Inline KGE value annotations in bold black
  for rect, val in zip(bars, vals):
    y_pos = val + 0.025 if val >= 0 else 0.025
    ax2.text(
        rect.get_x() + rect.get_width() / 2.0,
        y_pos,
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color="#000000",
        rotation=90,
        zorder=5,
    )

ax2.set_ylim(0.0, 1.18)
ax2.set_xticks(x)
ax2.set_xticklabels(
    stations, fontsize=12, fontweight="bold", color="#000000", rotation=20, ha="right"
)
ax2.set_xlabel(
    "Meteorological Station (Eritrea)",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)

# ------------------------------------------------------------------------------
# STEP 5: Shared Bold Legend & High-Resolution Export (680 DPI)
# ------------------------------------------------------------------------------
handles, labels = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
all_handles = handles + h2
all_labels = labels + l2

legend = fig.legend(
    all_handles,
    all_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=3,
    frameon=True,
    facecolor="#ffffff",
    edgecolor="#000000",
    fontsize=12,
)
legend.get_frame().set_linewidth(3.5)
for txt in legend.get_texts():
  txt.set_fontweight("bold")
  txt.set_color("#000000")
  txt.set_fontsize(12)

plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.91])

# Save at 680 DPI in both PNG and TIFF for journal submission
fig.savefig(
    PNG_OUT,
    dpi=680,
    facecolor="#ffffff",
    edgecolor="#000000",
    bbox_inches="tight",
)
fig.savefig(
    TIF_OUT,
    dpi=680,
    facecolor="#ffffff",
    edgecolor="#000000",
    bbox_inches="tight",
)
plt.show()

print(f"[DONE] Saved 680-DPI figures to:\n  -> {PNG_OUT}\n  -> {TIF_OUT}")

## 2. Historical CMIP6 & ERA5 Baseline Inventory (1981–2014)
This diagnostic step inventories all historical CMIP6 model files alongside the monthly ERA5 reference dataset in `1A_HISTORICAL_RAW` (1981–2014), verifies file dimensions and column schemas, and flags non-numeric strings before downscaling, bias correction, or model evaluation.

In [ ]:
# ==============================================================================
# SCRIPT 2.0: HISTORICAL CMIP6 MODEL INVENTORY & DATASET DIAGNOSTIC (1981-2014)
# Input Directory : /content/eritrea-cmip6-temperature-evaluation/1_DATA/1A_HISTORICAL_RAW
# Purpose         : Inventory CMIP6 models, inspect ERA5 reference schema, and
#                   screen for non-numeric entries prior to model evaluation.
# ==============================================================================

import glob
import os
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Historical Input Paths & Discover CMIP6 Models
# ------------------------------------------------------------------------------
RAW_DATA_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1A_HISTORICAL_RAW"
)
ERA5_REF_PATH = os.path.join(
    RAW_DATA_DIR, "ERA5_Temperature_Monthly_1981_2014.csv"
)
MODEL_PATTERN = "CMIP6_*_Temperature_Monthly_1981_2014.csv"

model_paths = sorted(glob.glob(os.path.join(RAW_DATA_DIR, MODEL_PATTERN)))
if not model_paths:
  raise FileNotFoundError(
      f"No CMIP6 files matching '{MODEL_PATTERN}' found in: {RAW_DATA_DIR}"
  )

model_names = [
    os.path.basename(p)
    .replace("CMIP6_", "")
    .replace("_Temperature_Monthly_1981_2014.csv", "")
    for p in model_paths
]

print("=" * 80)
print("TABLE 2.1: HISTORICAL CMIP6 MODEL INVENTORY (1981-2014)")
print("=" * 80)
print(f"[STEP 1] Total CMIP6 Models Detected: {len(model_names)}\n")
for idx, name in enumerate(model_names, start=1):
  print(f"  {idx:02d}. {name}")

# ------------------------------------------------------------------------------
# STEP 2: Inspect ERA5 Reference Dataset Structure, Dtypes & Sample Rows
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 2.2: ERA5 HISTORICAL REFERENCE STRUCTURE & DATA TYPES")
print("=" * 80)
if os.path.exists(ERA5_REF_PATH):
  era5_df = pd.read_csv(ERA5_REF_PATH)
  print(f"  • File Shape       : {era5_df.shape[0]} rows x {era5_df.shape[1]} columns")
  print(f"  • Columns          : {list(era5_df.columns)}")
  print("\n  • Column Data Types:")
  print(era5_df.dtypes.to_string())
  print("\n  • Preview (First 3 Rows):")
  print(era5_df.head(3).to_string())
else:
  print(f"  [WARNING] ERA5 reference file not found at: {ERA5_REF_PATH}")

# ------------------------------------------------------------------------------
# STEP 3: Inspect Sample CMIP6 Model & Scan for Non-Numeric Strings
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 2.3: SAMPLE CMIP6 MODEL SCHEMA & NON-NUMERIC VALUE AUDIT")
print("=" * 80)
sample_df = pd.read_csv(model_paths[0])
sample_model = model_names[0]

print(f"  • Sample Model     : {sample_model}")
print(f"  • File Shape       : {sample_df.shape[0]} rows x {sample_df.shape[1]} columns")
print(f"  • Columns          : {list(sample_df.columns)}")
print("\n  • Column Data Types:")
print(sample_df.dtypes.to_string())

# Exclude metadata/geometry columns before scanning numeric temperature fields
metadata_cols = {"date", "time", "system:index", ".geo"}
numeric_candidate_cols = [
    col for col in sample_df.columns if col.lower() not in metadata_cols
]

print("\n  • Non-Numeric String Audit Across Data Columns:")
for col in numeric_candidate_cols:
  non_numeric_mask = (
      pd.to_numeric(sample_df[col], errors="coerce").isna()
      & sample_df[col].notna()
  )
  bad_vals = sample_df.loc[non_numeric_mask, col].unique()
  if len(bad_vals) > 0:
    print(f"    [WARNING] '{col}' contains non-numeric strings: {bad_vals}")
  else:
    print(f"    [OK] '{col}' contains only numeric values or NaNs.")

## 3. Historical CMIP6 Model Evaluation & Comprehensive Rating Index (CRI)

### 3.1 Objective & Evaluation Scope
Using the validated monthly ERA5 dataset (`1981–2014`) as the observational baseline, this script evaluates each historical CMIP6 general circulation model across Eritrea's agro-ecological zones (`AGRO_ZONE`) and temperature variables ($T_{\text{mean}}$, $T_{\min}$, and $T_{\max}$).

### 3.2 Eight-Metric Statistical Evaluation Suite
For each `(Model, AGRO_ZONE, Variable)` combination, eight complementary performance indicators are computed on temporally aligned monthly pairs ($O_i$ = ERA5 baseline, $S_i$ = CMIP6 simulation):

1. **Pearson Correlation Coefficient ($r$):** Measures linear temporal and seasonal phase agreement ($-1 \le r \le 1$).
2. **Normalized Standard Deviation ($\text{NSD} = \sigma_S / \sigma_O$):** Evaluates relative amplitude fidelity (ideal value $= 1.0$).
3. **Centered Root Mean Square Difference ($\text{CRMSD}$, $^\circ\text{C}$):** Quantifies random pattern error after removing mean biases.
4. **Root Mean Square Error ($\text{RMSE}$, $^\circ\text{C}$):** Measures overall error magnitude in native temperature units.
5. **Mean Bias ($\text{MB}$, $^\circ\text{C}$):** Quantifies systematic warm ($+$) or cold ($-$) offsets ($\overline{S - O}$).
6. **Normalized Root Mean Square Error ($\text{NRMSE}$):** Scales $\text{RMSE}$ by the observed temperature range ($\max(O) - \min(O)$).
7. **Taylor Skill Score ($\text{TSS}$):** Combines correlation ($r$) and normalized variability ($\text{NSD}$) into a bounded skill score ($0 \le \text{TSS} \le 1$, with reference $r_0 = 0.999$).
8. **Interannual Variability Skill ($\text{IVS}$):** Measures variance distortion via $(\sigma_S/\sigma_O - \sigma_O/\sigma_S)^2$ (ideal value $= 0.0$).

### 3.3 Comprehensive Rating Index (CRI) Aggregation
Within each `(AGRO_ZONE, Variable)` stratum, all eight metrics are min–max normalized onto $[0, 1]$ so that $1.0$ always represents optimal model skill:
* **Maximized metrics ($r$, $\text{TSS}$):** $S = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$
* **Target-1 metric ($\text{NSD}$):** $S_{\text{NSD}} = 1 - \frac{|\text{NSD} - 1| - \min(|\text{NSD} - 1|)}{\max(|\text{NSD} - 1|) - \min(|\text{NSD} - 1|)}$
* **Minimized error metrics ($\text{CRMSD}$, $\text{RMSE}$, $\text{NRMSE}$, $\text{IVS}$, $|\text{MB}|$):** $S = 1 - \frac{X - X_{\min}}{X_{\max} - X_{\min}}$

Equal weights ($w_k = 0.125$) are assigned across the eight normalized scores to compute the **Comprehensive Rating Index ($\text{CRI}$)**, which is then averaged across all agro-ecological zones and variables to establish the overall model ranking.

In [ ]:
# ==============================================================================
# SCRIPT 3.0: HISTORICAL CMIP6 MODEL EVALUATION & CRI RANKING (1981-2014)
# Baseline Reference : ERA5_Temperature_Monthly_1981_2014.csv
# Evaluation Scope   : All Agro-Ecological Zones (AGRO_ZONE) x Tmean, Tmin, Tmax
# Statistical Suite  : r, NSD, CRMSD, RMSE, MB, NRMSE, TSS, IVS -> Composite CRI
# ==============================================================================

import glob
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Input/Output Directories & Load ERA5 Baseline
# ------------------------------------------------------------------------------
RAW_DATA_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1A_HISTORICAL_RAW"
)
OUTPUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/MODEL_EVALUATION_RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ERA5_PATH = os.path.join(RAW_DATA_DIR, "ERA5_Land_Temperature_Monthly_1981_2014.csv")
MODEL_PATTERN = "CMIP6_*_Temperature_Monthly_1981_2014.csv"
MODEL_PATHS = sorted(glob.glob(os.path.join(RAW_DATA_DIR, MODEL_PATTERN)))

if not os.path.exists(ERA5_PATH):
  raise FileNotFoundError(f"ERA5 baseline file not found at: {ERA5_PATH}")
if not MODEL_PATHS:
  raise FileNotFoundError(f"No CMIP6 model files found in: {RAW_DATA_DIR}")

era5_df = pd.read_csv(ERA5_PATH)
era5_df["date"] = era5_df["date"].astype(str)

print("=" * 88)
print("SCRIPT 3.0: CMIP6 HISTORICAL TEMPERATURE EVALUATION (ZONE & VARIABLE AWARE)")
print("=" * 88)
print(f"[STEP 1] Input Directory  : {RAW_DATA_DIR}")
print(f"         Output Directory : {OUTPUT_DIR}")
print(f"         Baseline Dataset : {os.path.basename(ERA5_PATH)}")
print(f"         CMIP6 Models     : {len(MODEL_PATHS)} models loaded")
print("=" * 88)


# ------------------------------------------------------------------------------
# STEP 2: Define Eight-Metric Statistical Formulations
# ------------------------------------------------------------------------------
def calc_pearson_r(obs: np.ndarray, sim: np.ndarray) -> float:
  """Pearson linear correlation coefficient (r)."""
  if np.std(obs) == 0 or np.std(sim) == 0:
    return np.nan
  return float(np.corrcoef(obs, sim)[0, 1])


def calc_nsd(obs: np.ndarray, sim: np.ndarray) -> float:
  """Normalized Standard Deviation (NSD = sigma_sim / sigma_obs)."""
  std_obs = np.std(obs, ddof=1)
  if std_obs == 0:
    return np.nan
  return float(np.std(sim, ddof=1) / std_obs)


def calc_crmsd(obs: np.ndarray, sim: np.ndarray) -> float:
  """Centered Root Mean Square Difference (CRMSD, °C)."""
  sim_centered = sim - np.mean(sim)
  obs_centered = obs - np.mean(obs)
  return float(np.sqrt(np.mean((sim_centered - obs_centered) ** 2)))


def calc_rmse(obs: np.ndarray, sim: np.ndarray) -> float:
  """Root Mean Square Error (RMSE, °C)."""
  return float(np.sqrt(np.mean((sim - obs) ** 2)))


def calc_mb(obs: np.ndarray, sim: np.ndarray) -> float:
  """Mean Bias (MB = mean(sim - obs), °C)."""
  return float(np.mean(sim - obs))


def calc_nrmse(obs: np.ndarray, sim: np.ndarray) -> float:
  """Range-Normalized Root Mean Square Error (NRMSE)."""
  denom = np.max(obs) - np.min(obs)
  if denom == 0:
    return np.nan
  return float(calc_rmse(obs, sim) / denom)


def calc_tss(r: float, nsd: float, r0: float = 0.999) -> float:
  """Taylor Skill Score (TSS, bounded in [0, 1])."""
  if np.isnan(r) or np.isnan(nsd) or nsd <= 0:
    return np.nan
  numerator = 4.0 * ((1.0 + r) ** 4)
  denominator = ((nsd + (1.0 / nsd)) ** 2) * ((1.0 + r0) ** 4)
  return float(numerator / denominator)


def calc_ivs(obs: np.ndarray, sim: np.ndarray) -> float:
  """Interannual Variability Skill (IVS = (sigma_sim/sigma_obs - sigma_obs/sigma_sim)^2)."""
  std_sim = np.std(sim, ddof=1)
  std_obs = np.std(obs, ddof=1)
  if std_obs == 0 or std_sim == 0:
    return np.nan
  return float(((std_sim / std_obs) - (std_obs / std_sim)) ** 2)


# ------------------------------------------------------------------------------
# STEP 3: Compute All 8 Evaluation Metrics Across Zones & Variables
# ------------------------------------------------------------------------------
print("\n[STEP 3] Computing 8 statistical metrics across all agro-ecological zones and variables...")

metrics_records = []
agro_zones = era5_df["AGRO_ZONE"].unique()
temp_vars = ["Tmean", "Tmin", "Tmax"]

for model_path in MODEL_PATHS:
  model_name = (
      os.path.basename(model_path)
      .replace("CMIP6_", "")
      .replace("_Temperature_Monthly_1981_2014.csv", "")
  )
  model_df = pd.read_csv(model_path)
  model_df["date"] = model_df["date"].astype(str)

  for zone in agro_zones:
    era5_zone = era5_df[era5_df["AGRO_ZONE"] == zone].sort_values("date")
    mod_zone = model_df[model_df["AGRO_ZONE"] == zone].sort_values("date")

    # Align monthly records by exact timestamp and agro-ecological zone
    merged = pd.merge(
        era5_zone, mod_zone, on=["date", "AGRO_ZONE"], suffixes=("_obs", "_sim")
    )

    for var in temp_vars:
      obs_col = f"{var}_obs"
      sim_col = f"{var}_sim"

      if obs_col not in merged.columns or sim_col not in merged.columns:
        continue

      obs = pd.to_numeric(merged[obs_col], errors="coerce").to_numpy(dtype=float)
      sim = pd.to_numeric(merged[sim_col], errors="coerce").to_numpy(dtype=float)

      valid_mask = ~np.isnan(obs) & ~np.isnan(sim)
      obs_clean = obs[valid_mask]
      sim_clean = sim[valid_mask]

      if len(obs_clean) == 0:
        continue

      r = calc_pearson_r(obs_clean, sim_clean)
      nsd = calc_nsd(obs_clean, sim_clean)
      crmsd = calc_crmsd(obs_clean, sim_clean)
      rmse = calc_rmse(obs_clean, sim_clean)
      mb = calc_mb(obs_clean, sim_clean)
      nrmse = calc_nrmse(obs_clean, sim_clean)
      tss = calc_tss(r, nsd)
      ivs = calc_ivs(obs_clean, sim_clean)

      metrics_records.append({
          "Model": model_name,
          "AGRO_ZONE": zone,
          "Variable": var,
          "n": int(len(obs_clean)),
          "r": r,
          "NSD": nsd,
          "CRMSD": crmsd,
          "RMSE": rmse,
          "MB": mb,
          "NRMSE": nrmse,
          "TSS": tss,
          "IVS": ivs,
      })

metrics_df = pd.DataFrame(metrics_records)

metrics_csv_path = os.path.join(OUTPUT_DIR, "1_CMIP6_8_Metrics_Summary.csv")
metrics_df.to_csv(metrics_csv_path, index=False)

print(f"  [OK] 8-metrics evaluation table exported to:\n       -> {metrics_csv_path}")

print("\n" + "=" * 88)
print("TABLE 3.1: SAMPLE CMIP6 EIGHT-METRIC EVALUATION OUTPUT (FIRST 10 ROWS)")
print("=" * 88)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(metrics_df.head(10).round(4).to_string(index=False))


# ------------------------------------------------------------------------------
# STEP 4: Normalize Sub-Metrics & Compute Comprehensive Rating Index (CRI)
# ------------------------------------------------------------------------------
print("\n[STEP 4] Normalizing sub-metrics per zone-variable stratum and computing CRI rankings...")


def compute_comprehensive_rating_index(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
  """Normalizes the 8 metrics per (AGRO_ZONE, Variable) and computes equal-weighted CRI."""
  cri_groups = []

  for (zone, var), group in df.groupby(["AGRO_ZONE", "Variable"], sort=False):
    g = group.copy()

    # 1. Maximized skill scores (higher is better)
    for col in ["r", "TSS"]:
      col_range = g[col].max() - g[col].min()
      g[f"S_{col.lower()}"] = (
          (g[col] - g[col].min()) / col_range if col_range != 0 else 0.0
      )

    # 2. Target-1 variability ratio (closer to 1.0 is better)
    nsd_dist = np.abs(g["NSD"] - 1.0)
    nsd_range = nsd_dist.max() - nsd_dist.min()
    g["S_nsd"] = (
        1.0 - ((nsd_dist - nsd_dist.min()) / nsd_range) if nsd_range != 0 else 1.0
    )

    # 3. Minimized error/dispersion metrics (lower is better)
    for col in ["CRMSD", "RMSE", "NRMSE", "IVS"]:
      col_range = g[col].max() - g[col].min()
      g[f"S_{col.lower()}"] = (
          1.0 - ((g[col] - g[col].min()) / col_range) if col_range != 0 else 1.0
      )

    # 4. Minimized absolute bias (closer to 0 °C is better)
    abs_mb = np.abs(g["MB"])
    mb_range = abs_mb.max() - abs_mb.min()
    g["S_mb"] = (
        1.0 - ((abs_mb - abs_mb.min()) / mb_range) if mb_range != 0 else 1.0
    )

    # 5. Equal-weighted composite CRI (w_k = 1/8 = 0.125)
    score_cols = [
        "S_r",
        "S_nsd",
        "S_crmsd",
        "S_rmse",
        "S_mb",
        "S_nrmse",
        "S_tss",
        "S_ivs",
    ]
    g["CRI"] = g[score_cols].mean(axis=1)
    cri_groups.append(g)

  detailed_cri = pd.concat(cri_groups, ignore_index=True)

  # Aggregate mean CRI across all agro-ecological zones and temperature variables
  summary_cri = (
      detailed_cri.groupby("Model", as_index=False)["CRI"]
      .mean()
      .sort_values(by="CRI", ascending=False)
      .reset_index(drop=True)
  )
  summary_cri.insert(0, "Rank", summary_cri.index + 1)

  return summary_cri, detailed_cri


summary_cri_df, detailed_cri_df = compute_comprehensive_rating_index(metrics_df)

# ------------------------------------------------------------------------------
# STEP 5: Export CRI Tables & Print Final Model Performance Rankings
# ------------------------------------------------------------------------------
cri_summary_path = os.path.join(OUTPUT_DIR, "2_CMIP6_CRI_Rankings_Summary.csv")
cri_detailed_path = os.path.join(
    OUTPUT_DIR, "3_CMIP6_CRI_Detailed_Per_Zone_Variable.csv"
)

summary_cri_df.to_csv(cri_summary_path, index=False)
detailed_cri_df.to_csv(cri_detailed_path, index=False)

print("\n" + "=" * 88)
print("TABLE 3.2: OVERALL CMIP6 MODEL PERFORMANCE RANKINGS (COMPOSITE CRI)")
print("=" * 88)
print(summary_cri_df.round(4).to_string(index=False))
print("=" * 88)

print("\n[STEP 5] Output files successfully exported:")
print(f"  -> 8-Metrics Table       : {metrics_csv_path}")
print(f"  -> Overall CRI Rankings  : {cri_summary_path}")
print(f"  -> Detailed CRI Breakdown: {cri_detailed_path}")

## 4. Agro-Ecological Zone Taylor Diagrams (CMIP6 vs. ERA5 Baseline)

### 4.1 Visual Diagnostic Purpose
To compare historical CMIP6 model skill across Eritrea's agro-ecological zones—**Arid Lowland (`ArL`)**, **Moist Highland (`MH`)**, and **Moist Lowland (`ML`)**—this script generates three 2×2 multi-panel Taylor diagrams (one each for $T_{\text{mean}}$, $T_{\min}$, and $T_{\max}$), placing a unified legend in the fourth quadrant.

### 4.2 Geometry & Skill Interpretation
* **Angular Axis:** Represents the Pearson correlation coefficient ($r$), where proximity to the right/vertical boundary indicates stronger temporal and seasonal phase alignment with ERA5.
* **Radial Axis:** Represents Normalized Standard Deviation ($\text{NSD} = \sigma_{\text{sim}} / \sigma_{\text{obs}}$), where the solid reference arc at $\text{NSD} = 1.0$ denotes matching variability amplitude and the red star marks the ERA5 reference point.
* **Dashed Contours:** Represent Centered Root Mean Square Difference ($\text{CRMSD}$) relative to the ERA5 reference point, capturing random pattern error independent of mean bias.

In [ ]:
# ==============================================================================
# SCRIPT 4.0: AGRO-ECOLOGICAL ZONE 2x2 TAYLOR DIAGRAMS (CMIP6 VS. ERA5)
# Input Metrics  : 1_CMIP6_8_Metrics_Summary.csv (fallback: CMIP6_8_Metrics_Summary.csv)
# Output Figures : Figure4_2x2_Taylor_{Tmean,Tmin,Tmax}.png & .tif (600 DPI)
# ==============================================================================

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Directories, Input Metrics & Journal Styling Rules
# ------------------------------------------------------------------------------
EVAL_DIR = "/content/eritrea-cmip6-temperature-evaluation/MODEL_EVALUATION_RESULTS"
FIGURES_DIR = os.path.join(EVAL_DIR, "FIGURES_TAYLOR_DIAGRAMS")
os.makedirs(FIGURES_DIR, exist_ok=True)

primary_csv = os.path.join(EVAL_DIR, "1_CMIP6_8_Metrics_Summary.csv")
fallback_csv = os.path.join(EVAL_DIR, "CMIP6_8_Metrics_Summary.csv")
METRICS_CSV = primary_csv if os.path.exists(primary_csv) else fallback_csv

if not os.path.exists(METRICS_CSV):
  raise FileNotFoundError(
      f"Metrics CSV not found at '{primary_csv}' or '{fallback_csv}'."
  )

# Strict publication typography and canvas contrast defaults
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

print("=" * 88)
print("SCRIPT 4.0: AGRO-ECOLOGICAL ZONE 2x2 TAYLOR DIAGRAMS")
print("=" * 88)
print(f"[STEP 1] Metrics Source   : {METRICS_CSV}")
print(f"         Output Directory : {FIGURES_DIR}")
print("=" * 88)

df_metrics = pd.read_csv(METRICS_CSV)

# Agro-ecological zones and panel ordering
ZONES_LIST = [
    ("ArL", "A", "Arid Lowland"),
    ("MH", "B", "Moist Highland"),
    ("ML", "C", "Moist Lowland"),
]

# Consistent model identity mapping across all panels and variables
models = sorted(df_metrics["Model"].dropna().unique())
num_models = len(models)

MARKERS = ["o", "s", "^", "D", "v", "p", "h", "8", "X", "d", "<", ">"]
CMAP = plt.get_cmap("tab20", max(num_models, 1))


# ------------------------------------------------------------------------------
# STEP 2: Define 2x2 Taylor Diagram Generator Function
# ------------------------------------------------------------------------------
def generate_aez_taylor_grid(
    df: pd.DataFrame, var_name: str, out_prefix: str
) -> None:
  """Generates a publication-grade 2x2 Taylor diagram grid for a single variable."""
  fig = plt.figure(figsize=(20, 18), dpi=600, facecolor="#ffffff")
  fig.patch.set_facecolor("#ffffff")

  legend_handles = []
  legend_labels = []

  for idx, (zone_code, panel_tag, zone_title) in enumerate(ZONES_LIST):
    sub_df = df[
        (df["AGRO_ZONE"] == zone_code) & (df["Variable"] == var_name)
    ].reset_index(drop=True)

    if sub_df.empty:
      print(
          f"  [WARNING] No records for Zone='{zone_code}', Variable='{var_name}'. Skipping panel."
      )
      continue

    ax = fig.add_subplot(2, 2, idx + 1, projection="polar")
    ax.set_facecolor("#ffffff")

    # Configure first-quadrant Taylor geometry (r from 0.0 to 1.0)
    ax.set_thetamin(0)
    ax.set_thetamax(90)

    # Angular correlation ticks (r)
    r_ticks = np.array([0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])
    ax.set_xticks(np.arccos(r_ticks))
    ax.set_xticklabels(
        [str(r) for r in r_ticks],
        fontsize=12,
        fontweight="bold",
        color="#000000",
    )

    # Radial NSD ticks and limits
    std_limit = 2.2
    ax.set_ylim(0, std_limit)
    ax.set_yticks([0.5, 1.0, 1.5, 2.0])
    ax.set_yticklabels(
        ["0.5", "1.0", "1.5", "2.0"],
        fontsize=12,
        fontweight="bold",
        color="#000000",
    )

    # Harden polar spine and tick appearance
    ax.spines["polar"].set_color("#000000")
    ax.spines["polar"].set_linewidth(3.5)
    ax.tick_params(
        axis="both",
        which="major",
        length=7,
        width=3.5,
        colors="#000000",
        labelsize=12,
    )
    ax.grid(
        True, linestyle="--", linewidth=1.5, color="#cccccc", alpha=0.85, zorder=1
    )

    # Reference NSD = 1.0 arc and ERA5 reference marker
    ax.plot(
        np.linspace(0, np.pi / 2.0, 200),
        np.ones(200),
        color="#000000",
        linewidth=4.5,
        linestyle="-",
        zorder=4,
    )
    ref_star, = ax.plot(
        0.0,
        1.0,
        marker="*",
        color="#D7191C",
        markersize=22,
        markeredgecolor="#000000",
        markeredgewidth=2.0,
        linestyle="",
        label="ERA5 (Reference)",
        clip_on=False,
        zorder=10,
    )

    # CRMSD contour arcs centered on ERA5 reference (r=1, NSD=1)
    rs, ts = np.meshgrid(
        np.linspace(0.0, std_limit, 150), np.linspace(0.0, np.pi / 2.0, 150)
    )
    crmsd_grid = np.sqrt(1.0 + rs**2 - 2.0 * rs * np.cos(ts))
    contours = ax.contour(
        ts,
        rs,
        crmsd_grid,
        levels=4,
        colors="#1F4E78",
        linestyles="--",
        linewidths=2.0,
        alpha=0.55,
        zorder=2,
    )
    clabels = ax.clabel(contours, inline=True, fontsize=11, fmt="%.2f")
    for lbl in clabels:
      lbl.set_fontweight("bold")
      lbl.set_color("#1F4E78")

    # Plot each CMIP6 model point
    for i, row in sub_df.iterrows():
      m_name = row["Model"]
      m_idx = models.index(m_name) if m_name in models else i

      r_val = float(np.clip(row["r"], -1.0, 1.0))
      nsd_val = float(row["NSD"])

      if np.isnan(r_val) or np.isnan(nsd_val):
        continue

      m_handle, = ax.plot(
          np.arccos(r_val),
          nsd_val,
          marker=MARKERS[m_idx % len(MARKERS)],
          linestyle="",
          markersize=12,
          markerfacecolor=CMAP(m_idx),
          markeredgecolor="#000000",
          markeredgewidth=2.0,
          label=m_name,
          zorder=8,
      )

      if idx == 0:
        legend_handles.append(m_handle)
        legend_labels.append(m_name)

    if idx == 0:
      legend_handles.insert(0, ref_star)
      legend_labels.insert(0, "ERA5 (Reference)")

    ax.set_title(
        f"({panel_tag}) {zone_title} — {var_name}",
        loc="left",
        fontsize=18,
        fontweight="bold",
        color="#000000",
        pad=32,
    )

  # Quadrant 4: Unified high-contrast legend panel
  leg_ax = fig.add_subplot(2, 2, 4)
  leg_ax.set_facecolor("#ffffff")
  leg_ax.axis("off")

  leg = leg_ax.legend(
      legend_handles,
      legend_labels,
      loc="center",
      fontsize=12,
      ncol=2,
      frameon=True,
      facecolor="#ffffff",
      edgecolor="#000000",
      title="CMIP6 Models & Baseline Reference",
      title_fontsize=14,
  )
  leg.get_frame().set_linewidth(3.5)
  leg.get_title().set_fontweight("bold")
  leg.get_title().set_color("#000000")
  for txt in leg.get_texts():
    txt.set_fontweight("bold")
    txt.set_color("#000000")
    txt.set_fontsize(12)

  # Global axis labels (14pt+ bold black)
  fig.text(
      0.50,
      0.03,
      "Correlation Coefficient (r)",
      ha="center",
      fontsize=16,
      fontweight="bold",
      color="#000000",
  )
  fig.text(
      0.03,
      0.50,
      "Normalized Standard Deviation (NSD)",
      va="center",
      rotation="vertical",
      fontsize=16,
      fontweight="bold",
      color="#000000",
  )

  plt.subplots_adjust(
      wspace=0.28, hspace=0.32, bottom=0.08, left=0.10, right=0.94, top=0.92
  )

  png_path = f"{out_prefix}.png"
  tif_path = f"{out_prefix}.tif"
  fig.savefig(
      png_path,
      dpi=600,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      tif_path,
      dpi=600,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  plt.show()
  plt.close(fig)

  print(f"  [OK] Exported 600-DPI Taylor grids:\n       -> {png_path}\n       -> {tif_path}")


# ------------------------------------------------------------------------------
# STEP 3: Batch Generate Taylor Grids for Tmean, Tmin, and Tmax
# ------------------------------------------------------------------------------
print("\n[STEP 3] Generating 2x2 Taylor diagrams across Tmean, Tmin, and Tmax...\n")

for temp_var in ["Tmean", "Tmin", "Tmax"]:
  output_prefix = os.path.join(FIGURES_DIR, f"Figure4_2x2_Taylor_{temp_var}")
  generate_aez_taylor_grid(df_metrics, temp_var, output_prefix)

print("\n" + "=" * 88)
print("[COMPLETE] All agro-ecological zone Taylor diagrams generated.")
print(f"Output Folder: {FIGURES_DIR}")
print("=" * 88)

## 5. Comprehensive Rating Index (CRI) & Interannual Variability Skill (IVS) Visual Diagnostics

### 5.1 Figure Suite Overview
This notebook generates three high-resolution (`680 DPI`) visual diagnostics summarizing historical CMIP6 model performance relative to the ERA5 baseline (`1981–2014`):

1. **Figure 5A — Overall Composite CRI Rankings:** Ranks all evaluated CMIP6 models from highest to lowest composite CRI, highlighting the **Top 10 Selected Models** (green), **Mid-Tier Models** (blue), and **Lowest 3 Models** (red) with inline numeric scores.
2. **Figure 5B — Zone-by-Variable CRI Performance Matrix:** Displays exact normalized CRI scores across each agro-ecological zone (`ArL`, `MH`, `ML`) and temperature variable ($T_{\text{mean}}$, $T_{\min}$, $T_{\max}$), ordered by overall model rank.
3. **Figure 5C — Multi-Panel Interannual Variability Skill (IVS):** Compares interannual variance fidelity across agro-ecological zones for $T_{\text{mean}}$, $T_{\min}$, and $T_{\max}$ (where $\text{IVS} \to 0$ denotes optimal variance reproduction).

In [ ]:
# ==============================================================================
# SCRIPT 5.0: CRI RANKINGS, AEZ PERFORMANCE HEATMAP & MULTI-PANEL IVS PLOTS
# Input CSVs     : 2_CMIP6_CRI_Rankings_Summary.csv
#                  3_CMIP6_CRI_Detailed_Per_Zone_Variable.csv
# Output Figures : Figure5A_Overall_CRI_Rankings.png/.tif (680 DPI)
#                  Figure5B_CRI_AEZ_Heatmap.png/.tif (680 DPI)
#                  Figure5C_IVS_AEZ_Evaluation.png/.tif (680 DPI)
# ==============================================================================

import os
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ------------------------------------------------------------------------------
# STEP 1: Configure Directories, Input Resolution & Journal Styling Rules
# ------------------------------------------------------------------------------
EVAL_DIR = "/content/eritrea-cmip6-temperature-evaluation/MODEL_EVALUATION_RESULTS"
FIGURES_DIR = os.path.join(EVAL_DIR, "FIGURES_CRI_IVS")
os.makedirs(FIGURES_DIR, exist_ok=True)


def resolve_csv(primary_name: str, fallback_name: str) -> str:
  p1 = os.path.join(EVAL_DIR, primary_name)
  p2 = os.path.join(EVAL_DIR, fallback_name)
  if os.path.exists(p1):
    return p1
  if os.path.exists(p2):
    return p2
  raise FileNotFoundError(
      f"Neither '{p1}' nor '{p2}' was found in {EVAL_DIR}."
  )


RANKINGS_CSV = resolve_csv(
    "2_CMIP6_CRI_Rankings_Summary.csv", "CMIP6_CRI_Rankings_Summary.csv"
)
DETAILED_CSV = resolve_csv(
    "3_CMIP6_CRI_Detailed_Per_Zone_Variable.csv", "CMIP6_8_Metrics_Summary.csv"
)

# Strict publication typography and canvas contrast defaults
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

print("=" * 88)
print("SCRIPT 5.0: CRI RANKINGS, AEZ HEATMAP & INTERANNUAL VARIABILITY SKILL (IVS)")
print("=" * 88)
print(f"[STEP 1] Rankings Source  : {RANKINGS_CSV}")
print(f"         Detailed Source  : {DETAILED_CSV}")
print(f"         Output Directory : {FIGURES_DIR}")
print("=" * 88)

df_rankings = pd.read_csv(RANKINGS_CSV)
df_detailed = pd.read_csv(DETAILED_CSV)

df_rankings.columns = df_rankings.columns.str.strip()
df_detailed.columns = df_detailed.columns.str.strip()

# Resolve CRI score column safely (excluding 'Rank')
cri_col = None
for candidate in ["CRI", "Overall_CRI", "CRI_Score", "Overall_Score", "Score"]:
  if candidate in df_rankings.columns:
    cri_col = candidate
    break

if cri_col is None:
  numeric_candidates = [
      c
      for c in df_rankings.select_dtypes(include=[np.number]).columns
      if c.lower() != "rank"
  ]
  if not numeric_candidates:
    raise KeyError(
        f"No numeric CRI score column found in {RANKINGS_CSV}. Columns: {list(df_rankings.columns)}"
    )
  cri_col = numeric_candidates[0]

print(f"  [OK] Active CRI score column: '{cri_col}'")


# ------------------------------------------------------------------------------
# STEP 2: Helper Functions for Hardened Spines, Ticks, Grid & Dual Export
# ------------------------------------------------------------------------------
def harden_cartesian_axis(ax: plt.Axes, grid_axis: str = "x") -> None:
  """Applies heavy black borders (3.5pt), thick ticks (3.5x7), and dashed grid."""
  ax.set_facecolor("#ffffff")
  ax.set_axisbelow(True)

  for side in ["top", "bottom", "left", "right"]:
    spine = ax.spines[side]
    spine.set_visible(True)
    spine.set_color("#000000")
    spine.set_linewidth(3.5)

  ax.tick_params(
      axis="both",
      which="major",
      direction="out",
      length=7,
      width=3.5,
      colors="#000000",
      labelsize=12,
  )
  for lbl in ax.get_xticklabels() + ax.get_yticklabels():
    lbl.set_fontweight("bold")
    lbl.set_color("#000000")
    lbl.set_fontsize(12)

  if grid_axis in ("x", "y", "both"):
    ax.grid(
        axis=grid_axis,
        which="major",
        linestyle="--",
        linewidth=1.5,
        color="#cccccc",
        zorder=0,
    )


def save_publication_figure(fig: plt.Figure, out_prefix: str) -> None:
  """Exports figure at 680 DPI in both PNG and TIFF formats."""
  png_path = f"{out_prefix}.png"
  tif_path = f"{out_prefix}.tif"
  fig.savefig(
      png_path,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      tif_path,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  plt.show()
  plt.close(fig)
  print(f"  [OK] Exported 680-DPI figure:\n       -> {png_path}\n       -> {tif_path}")


# ------------------------------------------------------------------------------
# STEP 3: Figure 5A — Overall Ranked CRI Horizontal Bar Chart
# ------------------------------------------------------------------------------
def plot_overall_cri_rankings(
    df_rank: pd.DataFrame, score_col: str, out_dir: str
) -> list[str]:
  """Plots horizontal ranked CRI bars with Top 10 highlighted in green."""
  df_sorted = df_rank.sort_values(by=score_col, ascending=True).reset_index(
      drop=True
  )
  n_models = len(df_sorted)

  bar_colors = []
  for idx in range(n_models):
    if idx >= n_models - 10:
      bar_colors.append("#1B7837")  # Top 10 Selected Models (Deep Green)
    elif idx < 3:
      bar_colors.append("#D7191C")  # Lowest 3 Models (Crimson Red)
    else:
      bar_colors.append("#2166AC")  # Mid-Tier Models (Cobalt Blue)

  fig, ax = plt.subplots(figsize=(13, 15), dpi=680, facecolor="#ffffff")
  fig.patch.set_facecolor("#ffffff")
  harden_cartesian_axis(ax, grid_axis="x")

  bars = ax.barh(
      df_sorted["Model"],
      df_sorted[score_col],
      color=bar_colors,
      edgecolor="#000000",
      linewidth=3.0,
      height=0.72,
      zorder=3,
  )

  # Inline bold score labels
  for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.008,
        bar.get_y() + bar.get_height() / 2.0,
        f"{width:.3f}",
        va="center",
        ha="left",
        fontsize=11,
        fontweight="bold",
        color="#000000",
        zorder=5,
    )

  ax.set_xlabel(
      "Comprehensive Rating Index (CRI)",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=10,
  )
  ax.set_ylabel(
      "CMIP6 Climate Models",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=10,
  )
  ax.set_xlim(0.0, float(df_sorted[score_col].max()) * 1.18)

  legend_elements = [
      Patch(
          facecolor="#1B7837",
          edgecolor="#000000",
          linewidth=2.5,
          label="Top 10 Selected Models",
      ),
      Patch(
          facecolor="#2166AC",
          edgecolor="#000000",
          linewidth=2.5,
          label="Mid-Tier Models",
      ),
      Patch(
          facecolor="#D7191C",
          edgecolor="#000000",
          linewidth=2.5,
          label="Lowest 3 Models",
      ),
  ]
  leg = ax.legend(
      handles=legend_elements,
      loc="lower right",
      fontsize=12,
      frameon=True,
      facecolor="#ffffff",
      edgecolor="#000000",
  )
  leg.get_frame().set_linewidth(3.5)
  for txt in leg.get_texts():
    txt.set_fontweight("bold")
    txt.set_color("#000000")
    txt.set_fontsize(12)

  plt.tight_layout()
  save_publication_figure(
      fig, os.path.join(out_dir, "Figure5A_Overall_CRI_Rankings")
  )

  # Return model names ordered from Rank 1 (top) to lowest (bottom)
  return df_sorted["Model"].iloc[::-1].tolist()


# ------------------------------------------------------------------------------
# STEP 4: Figure 5B — Zone-by-Variable CRI Performance Heatmap
# ------------------------------------------------------------------------------
def plot_cri_aez_heatmap(
    df_det: pd.DataFrame, ordered_models: list[str], out_dir: str
) -> None:
  """Plots annotated CRI heatmap across Agro-Ecological Zones and Variables."""
  if "CRI" in df_det.columns:
    val_col = "CRI"
    cbar_title = "Composite CRI Score"
  else:
    df_det = df_det.copy()
    df_det["Zone_Var_Score"] = df_det["r"] / (1.0 + df_det["RMSE"])
    val_col = "Zone_Var_Score"
    cbar_title = "Skill Score"

  pivot_df = df_det.pivot_table(
      index="Model", columns=["AGRO_ZONE", "Variable"], values=val_col
  )

  # Align model order to Figure 5A overall ranking
  valid_order = [m for m in ordered_models if m in pivot_df.index]
  pivot_df = pivot_df.reindex(index=valid_order)

  # Flatten MultiIndex columns into clean labels (e.g., "ArL | Tmean")
  pivot_df.columns = [f"{zone} | {var}" for zone, var in pivot_df.columns]

  fig, ax = plt.subplots(figsize=(15, 14), dpi=680, facecolor="#ffffff")
  fig.patch.set_facecolor("#ffffff")

  hm = sns.heatmap(
      pivot_df,
      cmap="YlGnBu",
      vmin=0.0,
      vmax=1.0,
      annot=True,
      fmt=".2f",
      linewidths=2.0,
      linecolor="#000000",
      cbar_kws={"label": cbar_title, "shrink": 0.85},
      ax=ax,
      annot_kws={"size": 10.5, "weight": "bold", "color": "#000000"},
  )

  # Harden heatmap outer spines and ticks
  for side in ["top", "bottom", "left", "right"]:
    spine = ax.spines[side]
    spine.set_visible(True)
    spine.set_color("#000000")
    spine.set_linewidth(3.5)

  ax.tick_params(
      axis="both",
      which="major",
      direction="out",
      length=7,
      width=3.5,
      colors="#000000",
      labelsize=12,
  )
  ax.set_xticklabels(
      ax.get_xticklabels(),
      rotation=30,
      ha="right",
      fontsize=12,
      fontweight="bold",
      color="#000000",
  )
  ax.set_yticklabels(
      ax.get_yticklabels(),
      rotation=0,
      fontsize=12,
      fontweight="bold",
      color="#000000",
  )

  # Style colorbar border, ticks, and bold 14pt label
  cbar = hm.collections[0].colorbar
  cbar.outline.set_edgecolor("#000000")
  cbar.outline.set_linewidth(3.5)
  cbar.ax.tick_params(
      length=7, width=3.5, colors="#000000", labelsize=12
  )
  for lbl in cbar.ax.get_yticklabels():
    lbl.set_fontweight("bold")
    lbl.set_color("#000000")
  cbar.set_label(
      cbar_title, fontsize=14, fontweight="bold", color="#000000", labelpad=12
  )

  ax.set_xlabel(
      "Agro-Ecological Zone & Temperature Variable",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=12,
  )
  ax.set_ylabel(
      "CMIP6 Climate Models (Ordered by Overall Rank)",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=10,
  )

  plt.tight_layout()
  save_publication_figure(fig, os.path.join(out_dir, "Figure5B_CRI_AEZ_Heatmap"))


# ------------------------------------------------------------------------------
# STEP 5: Figure 5C — Multi-Panel Interannual Variability Skill (IVS) Chart
# ------------------------------------------------------------------------------
def plot_ivs_multi_panel(
    df_det: pd.DataFrame, ordered_models: list[str], out_dir: str
) -> None:
  """Plots multi-panel IVS grouped bars across Agro-Ecological Zones."""
  df_plot = df_det.copy()
  if "IVS" not in df_plot.columns:
    df_plot["IVS"] = (df_plot["NSD"] - (1.0 / df_plot["NSD"])) ** 2

  variables = [
      v for v in ["Tmean", "Tmin", "Tmax"] if v in df_plot["Variable"].unique()
  ]
  if not variables:
    variables = list(df_plot["Variable"].unique()[:3])

  model_order = [m for m in ordered_models if m in df_plot["Model"].unique()]
  zone_order = sorted(df_plot["AGRO_ZONE"].unique())
  zone_palette = {
      "ArL": "#D95F02",  # Burnt orange
      "MH": "#1B7837",   # Deep green
      "ML": "#2166AC",   # Cobalt blue
  }
  colors = [zone_palette.get(z, "#7570B3") for z in zone_order]

  fig, axes = plt.subplots(
      nrows=len(variables),
      ncols=1,
      figsize=(18, 6.2 * len(variables)),
      sharex=True,
      dpi=680,
      facecolor="#ffffff",
  )
  fig.patch.set_facecolor("#ffffff")
  if len(variables) == 1:
    axes = [axes]

  panel_tags = ["(A)", "(B)", "(C)"]
  for idx, var in enumerate(variables):
    ax = axes[idx]
    harden_cartesian_axis(ax, grid_axis="y")

    sub_df = df_plot[df_plot["Variable"] == var]
    sns.barplot(
        data=sub_df,
        x="Model",
        y="IVS",
        hue="AGRO_ZONE",
        order=model_order,
        hue_order=zone_order,
        palette=colors,
        edgecolor="#000000",
        linewidth=2.5,
        ax=ax,
        zorder=3,
    )

    # Ideal IVS = 0.0 reference line
    ax.axhline(0.0, color="#000000", linewidth=4.0, zorder=4)

    # Panel tag inside top-left corner
    ax.text(
        0.012,
        0.94,
        f"{panel_tags[idx]} {var}",
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        color="#000000",
        ha="left",
        va="top",
        bbox=dict(
            boxstyle="square,pad=0.25",
            facecolor="#ffffff",
            edgecolor="#000000",
            linewidth=2.5,
        ),
        zorder=6,
    )

    ax.set_ylabel(
        "IVS Score (Lower = Better)",
        fontsize=14,
        fontweight="bold",
        color="#000000",
        labelpad=10,
    )
    ax.set_xlabel("")

    if idx == 0:
      leg = ax.legend(
          title="Agro-Ecological Zone",
          loc="upper right",
          fontsize=12,
          title_fontsize=13,
          frameon=True,
          facecolor="#ffffff",
          edgecolor="#000000",
      )
      leg.get_frame().set_linewidth(3.5)
      leg.get_title().set_fontweight("bold")
      leg.get_title().set_color("#000000")
      for txt in leg.get_texts():
        txt.set_fontweight("bold")
        txt.set_color("#000000")
    else:
      if ax.get_legend() is not None:
        ax.get_legend().remove()

  axes[-1].set_xlabel(
      "CMIP6 Climate Models (Ordered by Overall CRI Rank)",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=12,
  )
  axes[-1].tick_params(axis="x", rotation=75, labelsize=12)
  for lbl in axes[-1].get_xticklabels():
    lbl.set_fontweight("bold")
    lbl.set_color("#000000")

  plt.tight_layout()
  save_publication_figure(
      fig, os.path.join(out_dir, "Figure5C_IVS_AEZ_Evaluation")
  )


# ------------------------------------------------------------------------------
# STEP 6: Execute All Three CRI & IVS Visual Diagnostics
# ------------------------------------------------------------------------------
print("\n[STEP 3/6] Generating Figure 5A (Overall CRI Rankings Bar Chart)...")
ranked_model_order = plot_overall_cri_rankings(
    df_rankings, cri_col, FIGURES_DIR
)

print("\n[STEP 4/6] Generating Figure 5B (CRI Performance Heatmap Across AEZs)...")
plot_cri_aez_heatmap(df_detailed, ranked_model_order, FIGURES_DIR)

print("\n[STEP 5/6] Generating Figure 5C (Multi-Panel IVS Evaluation Plot)...")
plot_ivs_multi_panel(df_detailed, ranked_model_order, FIGURES_DIR)

print("\n" + "=" * 88)
print("[COMPLETE] All CRI and IVS figures exported at 680 DPI.")
print(f"Output Folder: {FIGURES_DIR}")
print("=" * 88)

## 6. Near-Future Top-10 CMIP6 & ERA5 Baseline Dataset Audit (2015–2026)

### 6.1 Purpose & Audit Scope
Following the historical model evaluation and selection of the top-performing CMIP6 models (`1981–2014`), this script audits the near-future (`2015–2026`) monthly temperature datasets stored in `1C_TOP_PERFORMING_MODEL_RAW` for both the **Top-10 CMIP6 ensemble/models** and the **ERA5 reference dataset**.

### 6.2 Quality Assurance Checks
Each dataset is inspected for row/column dimensions, schema consistency, unique models, shared socio-economic pathways (`Scenario`), agro-ecological zones (`AGRO_ZONE`), temporal coverage (`2015–2026`), missing values, and summary temperature distributions ($T_{\text{mean}}$, $T_{\min}$, and $T_{\max}$ in $^\circ\text{C}$) prior to near-future projection evaluation.

In [ ]:
# ==============================================================================
# SCRIPT 6.0: NEAR-FUTURE TOP-10 CMIP6 & ERA5 DATASET AUDIT (2015-2026)
# Input Directory : /content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW
# Purpose         : Audit schema, temporal coverage, missingness, and summary
#                   temperature statistics (°C) for 2015-2026 CMIP6 & ERA5 files.
# ==============================================================================

import os
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Near-Future Dataset Paths & Verify File Availability
# ------------------------------------------------------------------------------
NEAR_FUTURE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
CMIP6_NF_PATH = os.path.join(
    NEAR_FUTURE_DIR,
    "CMIP6_Top10_NearFuture_2015_2026_09_Monthly_Temperatures.csv",
)
ERA5_NF_PATH = os.path.join(
    NEAR_FUTURE_DIR, "ERA5_Land_NearFuture_2015_2026_09_Monthly_Temperatures.csv"
)

for path in (CMIP6_NF_PATH, ERA5_NF_PATH):
  if not os.path.exists(path):
    raise FileNotFoundError(f"Required near-future dataset not found: {path}")

df_cmip6_nf = pd.read_csv(CMIP6_NF_PATH)
df_era5_nf = pd.read_csv(ERA5_NF_PATH)

print("=" * 88)
print("SCRIPT 6.0: ERITREA CMIP6 TOP-10 VS. ERA5 NEAR-FUTURE (2015-2020) AUDIT")
print("=" * 88)
print(f"[STEP 1] Near-Future Directory : {NEAR_FUTURE_DIR}")
print(f"         CMIP6 Top-10 File     : {os.path.basename(CMIP6_NF_PATH)}")
print(f"         ERA5 Baseline File    : {os.path.basename(ERA5_NF_PATH)}")
print("=" * 88)


# ------------------------------------------------------------------------------
# STEP 2: Define Standardized Dataset Audit Function
# ------------------------------------------------------------------------------
def audit_near_future_dataset(
    df: pd.DataFrame, section_code: str, dataset_label: str
) -> None:
  """Prints structure, metadata breakdowns, missingness, and °C stats."""
  df_clean = df.copy()
  df_clean.columns = df_clean.columns.str.strip()

  print("\n" + "#" * 88)
  print(f"{section_code}: {dataset_label.upper()} DATASET AUDIT")
  print("#" * 88)

  print("\n  [A] Dimensions & Schema")
  print(f"      • Total Rows    : {df_clean.shape[0]}")
  print(f"      • Total Columns : {df_clean.shape[1]}")
  print(f"      • Columns       : {list(df_clean.columns)}")

  print("\n  [B] Sample Records (First 3 Rows)")
  print(df_clean.head(3).to_string(index=False))

  print("\n  [C] Metadata & Temporal Coverage")
  if "Model" in df_clean.columns:
    unique_models = sorted(df_clean["Model"].dropna().unique().tolist())
    print(f"      • Unique Models ({len(unique_models)}) : {unique_models}")
  if "Scenario" in df_clean.columns:
    print(
        f"      • Scenarios        : {sorted(df_clean['Scenario'].dropna().unique().tolist())}"
    )
  if "AGRO_ZONE" in df_clean.columns:
    print(
        f"      • Agro-Eco Zones : {sorted(df_clean['AGRO_ZONE'].dropna().unique().tolist())}"
    )

  # Support either 'Date' or 'date' column naming
  date_col = next(
      (c for c in ["Date", "date"] if c in df_clean.columns), None
  )
  if date_col is not None:
    dates_sorted = sorted(df_clean[date_col].astype(str).dropna().unique())
    print(
        f"      • Date Range       : {dates_sorted[0]} to {dates_sorted[-1]} "
        f"(Unique Months: {len(dates_sorted)})"
    )

  print("\n  [D] Missing Value Count per Column")
  null_counts = df_clean.isnull().sum()
  for col_name, cnt in null_counts.items():
    status = "OK" if cnt == 0 else "WARNING"
    print(f"      • [{status}] {col_name}: {int(cnt)} missing")

  print("\n  [E] Temperature Summary Statistics (°C)")
  temp_cols = [
      c for c in ["Tmean", "Tmin", "Tmax"] if c in df_clean.columns
  ]
  if temp_cols:
    temp_num = df_clean[temp_cols].apply(pd.to_numeric, errors="coerce")
    print(temp_num.describe().round(3).to_string())
  else:
    print("      • [WARNING] No Tmean, Tmin, or Tmax columns detected.")

  print("\n" + "=" * 88)


# ------------------------------------------------------------------------------
# STEP 3: Execute Audit for Top-10 CMIP6 & ERA5 Near-Future Datasets
# ------------------------------------------------------------------------------
print("\n[STEP 3] Running dataset audits for Top-10 CMIP6 and ERA5 (2015-2020)...")

audit_near_future_dataset(
    df_cmip6_nf,
    section_code="TABLE 6.1",
    dataset_label="Top-10 CMIP6 Near-Future Projections (2015-2020)",
)

audit_near_future_dataset(
    df_era5_nf,
    section_code="TABLE 6.2",
    dataset_label="ERA5 Near-Future Reference Baseline (2015-2020)",
)

print("[COMPLETE] Near-future dataset audit finished cleanly.")

## 7. Near-Future CMIP6 Top-10 Evaluation Against ERA5 (2015–2026)

### 7.1 Alignment & Error Metrics
Each top-10 CMIP6 model projection across `Scenario` and `AGRO_ZONE` is paired with the ERA5 reference series by `Date` and `AGRO_ZONE` for $T_{\text{mean}}$, $T_{\min}$, and $T_{\max}$ ($^\circ\text{C}$). Using valid non-missing monthly pairs ($n$), five evaluation metrics are computed:
* **Bias:** Mean simulation offset ($\overline{\text{Sim} - \text{Obs}}$, $^\circ\text{C}$).
* **MAE:** Mean absolute error ($\overline{|\text{Sim} - \text{Obs}|}$, $^\circ\text{C}$).
* **RMSE:** Root mean square error ($\sqrt{\overline{(\text{Sim} - \text{Obs})^2}}$, $^\circ\text{C}$).
* **Correlation ($r$):** Pearson linear correlation coefficient between simulated and ERA5 series.
* **IVS:** Interannual variability skill score, $(\sigma_{\text{Sim}}/\sigma_{\text{Obs}} - \sigma_{\text{Obs}}/\sigma_{\text{Sim}})^2$, where values closer to $0$ indicate stronger variability agreement.

In [ ]:
# ==============================================================================
# SCRIPT 7.0: NEAR-FUTURE TOP-10 CMIP6 VS. ERA5 EVALUATION METRICS (2015-2026)
# Input Datasets : CMIP6_Top10_NearFuture_2015_2020_Monthly_Temperatures.csv
#                  ERA5_NearFuture_2015_2026_Monthly_Temperatures.csv
# Output Table   : NearFuture_2015_2026_Evaluation_Metrics.csv
# ==============================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths & Load Near-Future Datasets
# ------------------------------------------------------------------------------
NEAR_FUTURE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
CMIP6_NF_PATH = os.path.join(
    NEAR_FUTURE_DIR,
    "CMIP6_Top10_NearFuture_2015_2026_09_Monthly_Temperatures.csv",
)
ERA5_NF_PATH = os.path.join(
    NEAR_FUTURE_DIR, "ERA5_Land_NearFuture_2015_2026_09_Monthly_Temperatures.csv"
)
OUTPUT_METRICS_CSV = os.path.join(
    NEAR_FUTURE_DIR, "NearFuture_2015_2026_Evaluation_Metrics.csv"
)

for path in (CMIP6_NF_PATH, ERA5_NF_PATH):
  if not os.path.exists(path):
    raise FileNotFoundError(f"Required input dataset not found: {path}")

df_cmip6 = pd.read_csv(CMIP6_NF_PATH)
df_era5 = pd.read_csv(ERA5_NF_PATH)

df_cmip6.columns = df_cmip6.columns.str.strip()
df_era5.columns = df_era5.columns.str.strip()

print("=" * 88)
print("SCRIPT 7.0: NEAR-FUTURE (2015-2026) CMIP6 TOP-10 EVALUATION METRICS")
print("=" * 88)
print(f"[STEP 1] CMIP6 Input  : {CMIP6_NF_PATH}")
print(f"         ERA5 Input   : {ERA5_NF_PATH}")
print(f"         Output Table : {OUTPUT_METRICS_CSV}")
print("=" * 88)

# ------------------------------------------------------------------------------
# STEP 2: Prepare ERA5 Reference & Merge by Date and Agro-Ecological Zone
# ------------------------------------------------------------------------------
print("\n[STEP 2] Aligning CMIP6 near-future series with ERA5 reference records...")

era5_obs = df_era5[["Date", "AGRO_ZONE", "Tmean", "Tmin", "Tmax"]].copy()
era5_obs["Date"] = era5_obs["Date"].astype(str)
era5_obs = era5_obs.rename(
    columns={"Tmean": "Tmean_obs", "Tmin": "Tmin_obs", "Tmax": "Tmax_obs"}
)

df_cmip6["Date"] = df_cmip6["Date"].astype(str)
df_merged = pd.merge(
    df_cmip6, era5_obs, on=["Date", "AGRO_ZONE"], how="inner"
)

if df_merged.empty:
  raise ValueError(
      "Merged near-future dataset is empty. Verify matching 'Date' and 'AGRO_ZONE' values."
  )

print(f"  [OK] Merged records: {len(df_merged)} rows across {df_merged['Model'].nunique()} models.")


# ------------------------------------------------------------------------------
# STEP 3: Define Metric Computation Function (Pairwise Clean)
# ------------------------------------------------------------------------------
def compute_near_future_metrics(
    group: pd.DataFrame, sim_col: str, obs_col: str
) -> pd.Series:
  """Computes Bias, MAE, RMSE, Pearson r, and IVS on valid monthly pairs."""
  sim_raw = pd.to_numeric(group[sim_col], errors="coerce").to_numpy(dtype=float)
  obs_raw = pd.to_numeric(group[obs_col], errors="coerce").to_numpy(dtype=float)

  valid = ~np.isnan(sim_raw) & ~np.isnan(obs_raw)
  sim = sim_raw[valid]
  obs = obs_raw[valid]

  n_valid = int(len(sim))
  if n_valid == 0:
    return pd.Series({
        "n": 0,
        "Bias": np.nan,
        "MAE": np.nan,
        "RMSE": np.nan,
        "Correlation": np.nan,
        "IVS": np.nan,
    })

  diff = sim - obs
  bias = float(np.mean(diff))
  mae = float(np.mean(np.abs(diff)))
  rmse = float(np.sqrt(np.mean(diff**2)))

  if n_valid > 1 and np.std(sim, ddof=1) > 0 and np.std(obs, ddof=1) > 0:
    r = float(np.corrcoef(sim, obs)[0, 1])
  else:
    r = np.nan

  if n_valid > 1:
    std_sim = float(np.std(sim, ddof=1))
    std_obs = float(np.std(obs, ddof=1))
    if std_sim > 0 and std_obs > 0:
      ratio = std_sim / std_obs
      ivs = float((ratio - (1.0 / ratio)) ** 2)
    else:
      ivs = np.nan
  else:
    ivs = np.nan

  return pd.Series({
      "n": n_valid,
      "Bias": bias,
      "MAE": mae,
      "RMSE": rmse,
      "Correlation": r,
      "IVS": ivs,
  })


# ------------------------------------------------------------------------------
# STEP 4: Evaluate Tmean, Tmin, and Tmax Across Model, Scenario & Agro-Zone
# ------------------------------------------------------------------------------
print("\n[STEP 4] Calculating near-future evaluation metrics for Tmean, Tmin, and Tmax...")

metrics_frames = []
variable_pairs = [
    ("Tmean", "Tmean_obs"),
    ("Tmin", "Tmin_obs"),
    ("Tmax", "Tmax_obs"),
]
group_cols = ["Model", "Scenario", "AGRO_ZONE"]

for sim_var, obs_var in variable_pairs:
  if sim_var not in df_merged.columns or obs_var not in df_merged.columns:
    print(f"  [WARNING] Missing column pair ({sim_var}, {obs_var}). Skipping.")
    continue

  grouped = (
      df_merged.groupby(group_cols, as_index=False, dropna=False)
      .apply(
          lambda g: compute_near_future_metrics(g, sim_var, obs_var),
          include_groups=False,
      )
      .reset_index(drop=True)
  )
  grouped["Variable"] = sim_var
  metrics_frames.append(grouped)

if not metrics_frames:
  raise RuntimeError("No near-future metrics were computed.")

df_evaluation = pd.concat(metrics_frames, ignore_index=True)
ordered_cols = [
    "Variable",
    "Model",
    "Scenario",
    "AGRO_ZONE",
    "n",
    "Bias",
    "MAE",
    "RMSE",
    "Correlation",
    "IVS",
]
df_evaluation = df_evaluation[ordered_cols]

# ------------------------------------------------------------------------------
# STEP 5: Export Near-Future Evaluation Metrics Table
# ------------------------------------------------------------------------------
df_evaluation.to_csv(OUTPUT_METRICS_CSV, index=False)

print("\n" + "=" * 88)
print("TABLE 7.1: SAMPLE NEAR-FUTURE (2015-2026) EVALUATION METRICS (FIRST 10 ROWS)")
print("=" * 88)
print(df_evaluation.head(10).round(4).to_string(index=False))
print("=" * 88)
print(f"\n[COMPLETE] Exported near-future evaluation table to:\n  -> {OUTPUT_METRICS_CSV}")

### 8. Near-Future CRI Ranking & Top-5 Ensemble Selection per Scenario
To rank the 10 near-future models within each `(Variable, AGRO_ZONE)` stratum, absolute bias (`|Bias|`), `MAE`, `RMSE`, and `IVS` are min–max scaled onto $[0, 1]$ (where $0$ is best), while correlation (`r`) is reverse-scaled onto $[0, 1]$ so lower values consistently indicate stronger agreement. Averaging the five normalized components yields the near-future **Comprehensive Rating Index ($\text{CRI}_{\text{NF}}$)**, which is then averaged across variables and agro-ecological zones per scenario to rank models from best (lowest $\text{CRI}_{\text{NF}}$) to worst and select the top 5 models per scenario for future projection extraction.

In [13]:
# ==============================================================================
# SCRIPT 8.0: NEAR-FUTURE MODEL CRI RANKING & BEST-VS.-WORST ENSEMBLE SELECTION
# Input Metrics  : NearFuture_2015_2026_Evaluation_Metrics.csv
# Output Summary : Model_Performance_Best_vs_Worst_Summary.csv
# ==============================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths & Load Near-Future Evaluation Metrics
# ------------------------------------------------------------------------------
NEAR_FUTURE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
METRICS_CSV = os.path.join(
    NEAR_FUTURE_DIR, "NearFuture_2015_2026_Evaluation_Metrics.csv"
)
SUMMARY_CSV = os.path.join(
    NEAR_FUTURE_DIR, "Model_Performance_Best_vs_Worst_Summary.csv"
)

if not os.path.exists(METRICS_CSV):
  raise FileNotFoundError(
      f"Near-future metrics file not found at: {METRICS_CSV}"
  )

df = pd.read_csv(METRICS_CSV)
df.columns = df.columns.str.strip()

print("=" * 88)
print("SCRIPT 8.0: NEAR-FUTURE MODEL CRI RANKING & TOP-5 ENSEMBLE SELECTION")
print("=" * 88)
print(f"[STEP 1] Metrics Input   : {METRICS_CSV}")
print(f"         Summary Output  : {SUMMARY_CSV}")
print("=" * 88)


# ------------------------------------------------------------------------------
# STEP 2: Compute Normalized Error-Distance CRI per Variable & Agro-Zone
# ------------------------------------------------------------------------------
def calculate_nf_cri(df_sub: pd.DataFrame) -> pd.DataFrame:
  """Normalizes 5 metrics onto [0, 1] (0 = optimal) within each stratum."""
  sub = df_sub.copy()
  sub["Abs_Bias"] = sub["Bias"].abs()

  for col in ["Abs_Bias", "MAE", "RMSE", "IVS"]:
    min_v = sub[col].min()
    max_v = sub[col].max()
    if pd.notna(min_v) and pd.notna(max_v) and max_v != min_v:
      sub[f"Norm_{col}"] = (sub[col] - min_v) / (max_v - min_v)
    else:
      sub[f"Norm_{col}"] = 0.0

  min_r = sub["Correlation"].min()
  max_r = sub["Correlation"].max()
  if pd.notna(min_r) and pd.notna(max_r) and max_r != min_r:
    sub["Norm_Corr"] = (max_r - sub["Correlation"]) / (max_r - min_r)
  else:
    sub["Norm_Corr"] = 0.0

  norm_cols = [
      "Norm_Abs_Bias",
      "Norm_MAE",
      "Norm_RMSE",
      "Norm_IVS",
      "Norm_Corr",
  ]
  sub["CRI"] = sub[norm_cols].mean(axis=1)
  return sub


print("\n[STEP 2] Computing near-future CRI per (Variable, AGRO_ZONE) stratum...")
df_ranked = df.groupby(["Variable", "AGRO_ZONE"], group_keys=False).apply(
    calculate_nf_cri, include_groups=False
)

# Restore grouping keys explicitly for complete traceability
for meta_col in ["Variable", "AGRO_ZONE"]:
  if meta_col not in df_ranked.columns:
    df_ranked[meta_col] = df.loc[df_ranked.index, meta_col]

# ------------------------------------------------------------------------------
# STEP 3: Aggregate Mean CRI by Model & Scenario and Classify Top 5 vs. Bottom 5
# ------------------------------------------------------------------------------
print("[STEP 3] Aggregating model CRI scores by scenario and assigning ranks...")

summary = (
    df_ranked.groupby(["Model", "Scenario"], as_index=False)["CRI"]
    .mean()
    .sort_values(by=["Scenario", "CRI", "Model"], ascending=[True, True, True])
    .reset_index(drop=True)
)
summary["Rank"] = summary.groupby("Scenario").cumcount() + 1
summary["Performance_Class"] = np.where(
    summary["Rank"] <= 5,
    "BEST (Retained for Ensemble)",
    "WORST (Dropped)",
)

summary = summary[["Scenario", "Rank", "Model", "CRI", "Performance_Class"]]

# ------------------------------------------------------------------------------
# STEP 4: Export Summary Table & Display Selected Scenario Ensembles
# ------------------------------------------------------------------------------
summary.to_csv(SUMMARY_CSV, index=False)

ssp245_mask = summary["Scenario"].astype(str).str.upper().str.replace("-", "", regex=False) == "SSP245"
ssp585_mask = summary["Scenario"].astype(str).str.upper().str.replace("-", "", regex=False) == "SSP585"

best_ssp245 = summary[ssp245_mask & (summary["Rank"] <= 5)]["Model"].tolist()
best_ssp585 = summary[ssp585_mask & (summary["Rank"] <= 5)]["Model"].tolist()

print("\n" + "=" * 88)
print("TABLE 8.1: NEAR-FUTURE CMIP6 MODEL RANKING (BEST TO WORST BY SCENARIO)")
print("=" * 88)
print(summary.round(4).to_string(index=False))

print("\n" + "=" * 88)
print("TABLE 8.2: TOP-5 ENSEMBLE MODELS SELECTED FOR FUTURE PROJECTIONS")
print("=" * 88)
print(f"  • Best 5 Models (SSP2-4.5) : {best_ssp245}")
print(f"  • Best 5 Models (SSP5-8.5) : {best_ssp585}")
print("=" * 88)
print(f"\n[COMPLETE] Exported model ranking summary to:\n  -> {SUMMARY_CSV}")

SCRIPT 8.0: NEAR-FUTURE MODEL CRI RANKING & TOP-5 ENSEMBLE SELECTION
[STEP 1] Metrics Input   : /content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW/NearFuture_2015_2026_Evaluation_Metrics.csv
         Summary Output  : /content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW/Model_Performance_Best_vs_Worst_Summary.csv

[STEP 2] Computing near-future CRI per (Variable, AGRO_ZONE) stratum...
[STEP 3] Aggregating model CRI scores by scenario and assigning ranks...

TABLE 8.1: NEAR-FUTURE CMIP6 MODEL RANKING (BEST TO WORST BY SCENARIO)
Scenario  Rank         Model    CRI            Performance_Class
  SSP245     1    KACE-1-0-G 0.2132 BEST (Retained for Ensemble)
  SSP245     2         NESM3 0.3085 BEST (Retained for Ensemble)
  SSP245     3 MPI-ESM1-2-LR 0.3091 BEST (Retained for Ensemble)
  SSP245     4 ACCESS-ESM1-5 0.3799 BEST (Retained for Ensemble)
  SSP245     5   BCC-CSM2-MR 0.4470 BEST (Retained for Ensemble)
  SSP245     6 MPI-

## 9. Core-3 CMIP6 Historical Baseline (1985–2014), Future Anomalies & Multi-Model Ensemble (2026–2100)

### 9.1 Baseline Climatology & Diurnal Temperature Range (DTR)
Using the historical Core-3 CMIP6 monthly datasets (`1985–2014`), we compute calendar-month climatological baselines (`Month` = $1, \dots, 12$) per `(Model, AGRO_ZONE)` for mean temperature ($T_{\text{mean}}$), minimum temperature ($T_{\min}$), maximum temperature ($T_{\max}$), and Diurnal Temperature Range ($\text{DTR} = T_{\max} - T_{\min}$).

### 9.2 Future Anomalies & Multi-Model Ensemble (MME) Spread (2026–2100)
Future monthly projections (`2026–2100`) are matched by `(Model, AGRO_ZONE, Month)` to calculate model-specific temperature and DTR anomalies relative to the `1985–2014` baseline ($\Delta X = X_{\text{fut}} - \overline{X}_{\text{base}}$). Across the Core-3 models, we then aggregate the Multi-Model Ensemble mean (`_mme`), inter-model standard deviation (`_std`), and range (`_min`, `_max`) by `Scenario`, `Period`, `Date`, `AGRO_ZONE`, and `Month`.

In [ ]:
# ==============================================================================
# SCRIPT 9.0: CORE-3 CMIP6 BASELINE (1985-2014), ANOMALIES & MME (2026-2100)
# Input Datasets : CMIP6_Core3_Historical_1985_2014_Monthly_Temperatures.csv
#                  CMIP6_Core3_Future_2026_2100_Monthly_Temperatures.csv
# Output Table   : Core3_MME_Monthly_Anomalies_2026_2100.csv
# ==============================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Directories, Paths & Verify Input File Availability
# ------------------------------------------------------------------------------
BASE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
os.makedirs(BASE_DIR, exist_ok=True)

HIST_PATH = os.path.join(
    BASE_DIR, "CMIP6_Core3_Historical_1985_2014_Monthly_Temperatures.csv"
)
FUT_PATH = os.path.join(
    BASE_DIR, "CMIP6_Core3_Future_2026_2100_Monthly_Temperatures.csv"
)
OUTPUT_MME_CSV = os.path.join(
    BASE_DIR, "Core3_MME_Monthly_Anomalies_2026_2100.csv"
)

for path, label in [(HIST_PATH, "Historical"), (FUT_PATH, "Future")]:
  if not os.path.exists(path):
    raise FileNotFoundError(
        f"[ERROR] {label} dataset not found at: {path}\n"
        f"Ensure the CSV is placed in: {BASE_DIR}"
    )

print("=" * 88)
print("SCRIPT 9.0: CORE-3 CMIP6 BASELINE CLIMATOLOGY, ANOMALIES & MME")
print("=" * 88)
print(f"[STEP 1] Workspace Directory : {BASE_DIR}")
print(f"         Historical Input    : {os.path.basename(HIST_PATH)}")
print(f"         Future Input        : {os.path.basename(FUT_PATH)}")
print(f"         Output MME Table    : {os.path.basename(OUTPUT_MME_CSV)}")
print("=" * 88)

# ------------------------------------------------------------------------------
# STEP 2: Load Datasets, Extract Calendar Month & Compute DTR
# ------------------------------------------------------------------------------
print(
    "\n[STEP 2] Loading historical and future Core-3 datasets and computing DTR..."
)

df_hist = pd.read_csv(HIST_PATH)
df_fut = pd.read_csv(FUT_PATH)

df_hist.columns = df_hist.columns.str.strip()
df_fut.columns = df_fut.columns.str.strip()


def prepare_temperature_frame(df: pd.DataFrame) -> pd.DataFrame:
  """Standardizes Date/Month, coerces Tmean/Tmin/Tmax, and computes DTR."""
  out = df.copy()
  out["Date"] = out["Date"].astype(str).str.strip()
  out["Month"] = out["Date"].apply(lambda x: int(x.split("-")[1]))

  for col in ["Tmean", "Tmin", "Tmax"]:
    if col in out.columns:
      out[col] = pd.to_numeric(out[col], errors="coerce")

  out["DTR"] = out["Tmax"] - out["Tmin"]
  return out


df_hist = prepare_temperature_frame(df_hist)
df_fut = prepare_temperature_frame(df_fut)

# ------------------------------------------------------------------------------
# STEP 3: Compute Monthly Climatological Baseline (1985-2014)
# ------------------------------------------------------------------------------
print(
    "[STEP 3] Computing 1985-2014 multi-year monthly baseline per Model & AGRO_ZONE..."
)

base_vars = ["Tmean", "Tmin", "Tmax", "DTR"]
baseline = (
    df_hist.groupby(["Model", "AGRO_ZONE", "Month"], as_index=False)[base_vars]
    .mean()
    .rename(
        columns={
            "Tmean": "Tmean_base",
            "Tmin": "Tmin_base",
            "Tmax": "Tmax_base",
            "DTR": "DTR_base",
        }
    )
)

# ------------------------------------------------------------------------------
# STEP 4: Merge Baseline with Future Projections & Calculate Model Anomalies
# ------------------------------------------------------------------------------
print(
    "[STEP 4] Merging historical baseline with future series and computing anomalies..."
)

df_merged = pd.merge(
    df_fut, baseline, on=["Model", "AGRO_ZONE", "Month"], how="left"
)

df_merged["dTmean"] = df_merged["Tmean"] - df_merged["Tmean_base"]
df_merged["dTmin"] = df_merged["Tmin"] - df_merged["Tmin_base"]
df_merged["dTmax"] = df_merged["Tmax"] - df_merged["Tmax_base"]
df_merged["dDTR"] = df_merged["DTR"] - df_merged["DTR_base"]

# ------------------------------------------------------------------------------
# STEP 5: Aggregate Multi-Model Ensemble (MME) Mean, Spread & Range (2026-2100)
# ------------------------------------------------------------------------------
print(
    "[STEP 5] Aggregating Core-3 Multi-Model Ensemble (MME) statistics..."
)

group_keys = ["Scenario", "Period", "Date", "AGRO_ZONE", "Month"]
mme_df = (
    df_merged.groupby(group_keys, as_index=False, dropna=False)
    .agg(
        Tmean_mme=("Tmean", "mean"),
        Tmin_mme=("Tmin", "mean"),
        Tmax_mme=("Tmax", "mean"),
        DTR_mme=("DTR", "mean"),
        dTmean_mme=("dTmean", "mean"),
        dTmin_mme=("dTmin", "mean"),
        dTmax_mme=("dTmax", "mean"),
        dDTR_mme=("dDTR", "mean"),
        dTmean_std=("dTmean", "std"),
        dTmean_min=("dTmean", "min"),
        dTmean_max=("dTmean", "max"),
        dTmin_std=("dTmin", "std"),
        dTmax_std=("dTmax", "std"),
    )
    .sort_values(by=["Scenario", "AGRO_ZONE", "Date"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------------------------
# STEP 6: Export MME Anomaly Dataset & Print Diagnostic Summaries
# ------------------------------------------------------------------------------
mme_df.to_csv(OUTPUT_MME_CSV, index=False)

print("\n" + "=" * 88)
print("TABLE 9.1: SAMPLE HISTORICAL BASELINE CLIMATOLOGY (1985-2014, FIRST 6 ROWS)")
print("=" * 88)
print(baseline.head(6).round(3).to_string(index=False))

print("\n" + "=" * 88)
print("TABLE 9.2: SAMPLE CORE-3 MME MONTHLY ANOMALIES (2026-2100, FIRST 10 ROWS)")
print("=" * 88)
preview_cols = [
    "Scenario",
    "Period",
    "Date",
    "AGRO_ZONE",
    "dTmean_mme",
    "dTmin_mme",
    "dTmax_mme",
    "dDTR_mme",
    "dTmean_std",
]
print(mme_df[preview_cols].head(10).round(3).to_string(index=False))
print("=" * 88)
print(f"\n[COMPLETE] Exported Core-3 MME anomalies table to:\n  -> {OUTPUT_MME_CSV}")

## 10. Future Projection Statistical Summaries & Decadal Warming Rates (2026–2100)

### 10.1 Period-Wise Warming & Night-vs.-Day Asymmetry
Using the Core-3 Multi-Model Ensemble (`MME`) monthly anomaly series (`Core3_MME_Monthly_Anomalies_2026_2100.csv`), we aggregate mean projected anomalies (`dTmean`, `dTmin`, `dTmax`, `dDTR`) and mean inter-model ensemble spread (`dTmean_Spread`) across each `Scenario`, `Period`, and `AGRO_ZONE`. We also compute `Night_vs_Day_Diff` ($\Delta T_{\min} - \Delta T_{\max}$, $^\circ\text{C}$) to capture asymmetric nocturnal versus daytime warming.

### 10.2 Annualized Decadal Warming Rates (°C / Decade)
Annual mean anomalies are aggregated per `(Scenario, AGRO_ZONE, Year)` across `2026–2100`, and linear least-squares regression slopes ($^\circ\text{C}/\text{yr}$) are scaled by $10$ to report decadal warming and DTR trend rates ($^\circ\text{C}/\text{decade}$).

In [ ]:
# ==============================================================================
# SCRIPT 10.0: FUTURE PROJECTION PERIOD WARMING & DECADAL TREND RATES (2026-2100)
# Input Dataset : Core3_MME_Monthly_Anomalies_2026_2100.csv
# Output Tables : Future_Period_Warming_Summary.csv
#                 Future_Decadal_Warming_Rates.csv
# ==============================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths, Verify MME Input & Load Dataset
# ------------------------------------------------------------------------------
BASE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
MME_PATH = os.path.join(BASE_DIR, "Core3_MME_Monthly_Anomalies_2026_2100.csv")
SUMMARY_CSV = os.path.join(BASE_DIR, "Future_Period_Warming_Summary.csv")
TREND_CSV = os.path.join(BASE_DIR, "Future_Decadal_Warming_Rates.csv")

if not os.path.exists(MME_PATH):
  raise FileNotFoundError(
      f"[ERROR] Required MME anomaly file not found at: {MME_PATH}\n"
      "Run Script 9.0 first to generate Core3_MME_Monthly_Anomalies_2026_2100.csv."
  )

print("=" * 88)
print("SCRIPT 10.0: FUTURE WARMING SUMMARIES & DECADAL TREND RATES (2026-2100)")
print("=" * 88)
print(f"[STEP 1] Input MME Table  : {MME_PATH}")
print(f"         Period Summary Out : {SUMMARY_CSV}")
print(f"         Decadal Trends Out : {TREND_CSV}")
print("=" * 88)

df_mme = pd.read_csv(MME_PATH)
df_mme.columns = df_mme.columns.str.strip()
df_mme["Date"] = df_mme["Date"].astype(str).str.strip()
df_mme["Year"] = df_mme["Date"].apply(lambda x: int(x.split("-")[0]))

anomaly_cols = [
    "dTmean_mme",
    "dTmin_mme",
    "dTmax_mme",
    "dDTR_mme",
    "dTmean_std",
]
for col in anomaly_cols:
  if col in df_mme.columns:
    df_mme[col] = pd.to_numeric(df_mme[col], errors="coerce")

# ------------------------------------------------------------------------------
# STEP 2: Compute Period-Wise Mean Warming & Nocturnal-Daytime Asymmetry
# ------------------------------------------------------------------------------
print(
    "\n[STEP 2] Computing period-wise mean anomalies and night-vs.-day warming difference..."
)

period_summary = (
    df_mme.groupby(["Scenario", "Period", "AGRO_ZONE"], as_index=False)
    .agg(
        dTmean=("dTmean_mme", "mean"),
        dTmin=("dTmin_mme", "mean"),
        dTmax=("dTmax_mme", "mean"),
        dDTR=("dDTR_mme", "mean"),
        dTmean_Spread=("dTmean_std", "mean"),
    )
    .reset_index(drop=True)
)
period_summary["Night_vs_Day_Diff"] = (
    period_summary["dTmin"] - period_summary["dTmax"]
)

# ------------------------------------------------------------------------------
# STEP 3: Compute Annualized Linear Decadal Trend Rates (°C / Decade)
# ------------------------------------------------------------------------------
print(
    "[STEP 3] Fitting linear annual trend slopes and scaling to °C/decade (2026-2100)..."
)

annual_mme = (
    df_mme.groupby(["Scenario", "AGRO_ZONE", "Year"], as_index=False)[
        ["dTmean_mme", "dTmin_mme", "dTmax_mme", "dDTR_mme"]
    ]
    .mean()
    .sort_values(by=["Scenario", "AGRO_ZONE", "Year"])
    .reset_index(drop=True)
)

trend_records = []
for (scenario, zone), grp in annual_mme.groupby(
    ["Scenario", "AGRO_ZONE"], sort=False
):
  years = grp["Year"].to_numpy(dtype=float)

  def decadal_slope(series_col: str) -> float:
    vals = grp[series_col].to_numpy(dtype=float)
    valid = ~np.isnan(years) & ~np.isnan(vals)
    if np.sum(valid) < 2 or np.std(years[valid]) == 0:
      return np.nan
    slope_per_year, _ = np.polyfit(years[valid], vals[valid], 1)
    return float(slope_per_year * 10.0)

  trend_records.append({
      "Scenario": scenario,
      "AGRO_ZONE": zone,
      "Rate_dTmean_per_decade": decadal_slope("dTmean_mme"),
      "Rate_dTmin_per_decade": decadal_slope("dTmin_mme"),
      "Rate_dTmax_per_decade": decadal_slope("dTmax_mme"),
      "Rate_dDTR_per_decade": decadal_slope("dDTR_mme"),
  })

df_trends = pd.DataFrame(trend_records)

# ------------------------------------------------------------------------------
# STEP 4: Export Statistical Summary Tables & Display Results
# ------------------------------------------------------------------------------
period_summary.to_csv(SUMMARY_CSV, index=False)
df_trends.to_csv(TREND_CSV, index=False)

print("\n" + "=" * 88)
print(
    "TABLE 10.1: PERIOD-WISE PROJECTED WARMING (°C RELATIVE TO 1985-2014 BASELINE)"
)
print("=" * 88)
print(period_summary.round(3).to_string(index=False))

print("\n" + "=" * 88)
print("TABLE 10.2: PROJECTED LINEAR DECADAL WARMING RATES (°C / DECADE, 2026-2100)")
print("=" * 88)
print(df_trends.round(3).to_string(index=False))
print("=" * 88)

print("\n[COMPLETE] Exported future projection statistical tables:")
print(f"  -> Period Summary : {SUMMARY_CSV}")
print(f"  -> Decadal Rates  : {TREND_CSV}")

## 11. Future Climate Projection Visualizations (2026–2100)

### 11.1 Six-Figure Future Warming Diagnostic Suite
Using the Core-3 Multi-Model Ensemble (`MME`) monthly anomaly series (`Core3_MME_Monthly_Anomalies_2026_2100.csv`), period-wise warming summary (`Future_Period_Warming_Summary.csv`), and decadal trend rates (`Future_Decadal_Warming_Rates.csv`), this script exports six high-contrast figures (`680 DPI` `.png`, `.tif`, and `.pdf`):

1. **Figure 11.1:** Long-term annual mean temperature anomaly trajectories (`ΔTmean`, `2026–2100`) with min–max model spread envelopes across `ArL`, `MH`, `ML`, and `OVERALL` for `SSP245` and `SSP585`.
2. **Figure 11.2:** Period-wise mean warming (`ΔTmean`) across agro-ecological zones and future periods with bold inline values.
3. **Figure 11.3:** Monthly seasonal warming cycles (`Jan–Dec`) across future periods for `SSP245` and `SSP585` (`OVERALL`).
4. **Figure 11.4:** Nighttime (`ΔTmin`) versus daytime (`ΔTmax`) asymmetric warming across future periods (`OVERALL`) with clean two-line labels and bold inline values.
5. **Figure 11.5:** Monthly Diurnal Temperature Range shift (`ΔDTR`) distributions across agro-ecological zones and scenarios relative to `0.0 °C`.
6. **Figure 11.6:** Linear decadal warming rates (`°C / decade`, `2026–2100`) across agro-ecological zones and scenarios with bold inline rates.

In [ ]:
# ==============================================================================
# SCRIPT 11.0 (REVISED): FUTURE CLIMATE VISUALIZATIONS (LABELS ABOVE FRAMES)
# Fix Applied    : Moved panel labels (a), (b), (c), (d) from inside the axes
#                  to above the top spine via ax.set_title(..., loc="left", pad=12)
#                  so they never overlap data curves, envelopes, or legends.
# ==============================================================================

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths, Verify Inputs & Enforce Strict Journal Styling Rules
# ------------------------------------------------------------------------------
BASE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
FIG_OUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/2_FIGURES"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

MME_PATH = os.path.join(BASE_DIR, "Core3_MME_Monthly_Anomalies_2026_2100.csv")
PERIOD_PATH = os.path.join(BASE_DIR, "Future_Period_Warming_Summary.csv")
TREND_PATH = os.path.join(BASE_DIR, "Future_Decadal_Warming_Rates.csv")

for path, label in [
    (MME_PATH, "MME Monthly Anomalies"),
    (PERIOD_PATH, "Period Warming Summary"),
    (TREND_PATH, "Decadal Warming Rates"),
]:
  if not os.path.exists(path):
    raise FileNotFoundError(f"[ERROR] Missing {label} file at: {path}")

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

df_mme = pd.read_csv(MME_PATH)
df_period = pd.read_csv(PERIOD_PATH)
df_trends = pd.read_csv(TREND_PATH)

for frame in (df_mme, df_period, df_trends):
  frame.columns = frame.columns.str.strip()

df_mme["Date"] = df_mme["Date"].astype(str).str.strip()
df_mme["Year"] = df_mme["Date"].apply(lambda x: int(x.split("-")[0]))

SSP_COLORS = {"SSP245": "#004C6D", "SSP585": "#B03A2E"}


# ------------------------------------------------------------------------------
# STEP 2: Helper Functions (Frame Header Above Axis + Spines + Grid + Export)
# ------------------------------------------------------------------------------
def apply_journal_axis_style(
    ax: plt.Axes, grid_axis: str = "both", frame_title: str | None = None
) -> None:
  """Applies heavy borders/ticks/grid and places frame_title ABOVE the top border."""
  ax.set_facecolor("#ffffff")
  ax.set_axisbelow(True)

  for side in ["top", "bottom", "left", "right"]:
    spine = ax.spines[side]
    spine.set_visible(True)
    spine.set_color("#000000")
    spine.set_linewidth(3.5)

  ax.tick_params(
      axis="both",
      which="major",
      direction="out",
      length=7,
      width=3.5,
      colors="#000000",
      labelsize=12,
  )
  for lbl in ax.get_xticklabels() + ax.get_yticklabels():
    lbl.set_fontweight("bold")
    lbl.set_color("#000000")
    lbl.set_fontsize(12)

  if grid_axis in ("x", "y", "both"):
    ax.grid(
        axis=grid_axis,
        which="major",
        linestyle="--",
        linewidth=1.5,
        color="#cccccc",
        zorder=0,
    )

  # Place frame label cleanly ABOVE the top border (outside the frame)
  if frame_title:
    ax.set_title(
        frame_title,
        loc="left",
        fontsize=14,
        fontweight="bold",
        color="#000000",
        pad=12,
    )


def style_bold_legend(ax: plt.Axes, loc: str = "upper left") -> None:
  """Formats legend box and text in pure bold black."""
  leg = ax.legend(
      loc=loc,
      frameon=True,
      facecolor="#ffffff",
      edgecolor="#000000",
      fontsize=12,
  )
  if leg is not None:
    leg.get_frame().set_linewidth(3.5)
    for txt in leg.get_texts():
      txt.set_fontweight("bold")
      txt.set_color("#000000")
      txt.set_fontsize(12)


def export_figure(fig: plt.Figure, out_prefix: str) -> None:
  """Exports figure at 680 DPI in PNG and TIFF plus vector PDF."""
  png_path = f"{out_prefix}.png"
  tif_path = f"{out_prefix}.tif"
  pdf_path = f"{out_prefix}.pdf"
  fig.savefig(
      png_path,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      tif_path,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      pdf_path,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  plt.show()
  plt.close(fig)
  print(f"  [OK] Exported:\n       -> {png_path}\n       -> {tif_path}")


def annotate_bars(
    ax: plt.Axes, fmt: str = "{:+.2f}", offset_ratio: float = 0.02, fontsize: float = 10.5
) -> None:
  """Places bold inline numeric labels above/below bar patches."""
  y_min, y_max = ax.get_ylim()
  span = max(y_max - y_min, 1.0)
  for patch in ax.patches:
    height = patch.get_height()
    if np.isnan(height):
      continue
    cx = patch.get_x() + patch.get_width() / 2.0
    if height >= 0:
      cy = height + span * offset_ratio
      va = "bottom"
    else:
      cy = height - span * offset_ratio
      va = "top"
    ax.text(
        cx,
        cy,
        fmt.format(height),
        ha="center",
        va=va,
        fontsize=fontsize,
        fontweight="bold",
        color="#000000",
        zorder=6,
    )


# ------------------------------------------------------------------------------
# STEP 3: Figure 11.1 — Long-Term Annual Anomaly Time Series (2026-2100)
# ------------------------------------------------------------------------------
print("\n[STEP 3] Generating Figure 11.1 (Long-Term Anomaly Time Series)...")

annual_mme = (
    df_mme.groupby(["Scenario", "AGRO_ZONE", "Year"], as_index=False)
    .agg(
        dTmean=("dTmean_mme", "mean"),
        dTmean_min=("dTmean_min", "mean"),
        dTmean_max=("dTmean_max", "mean"),
    )
    .sort_values(by=["Scenario", "AGRO_ZONE", "Year"])
)

zone_config = [
    ("ArL", "(a) Arid Lowland (ArL)"),
    ("MH", "(b) Moist Highland (MH)"),
    ("ML", "(c) Moist Lowland (ML)"),
    ("OVERALL", "(d) Overall Eritrea"),
]

fig, axes = plt.subplots(
    2, 2, figsize=(16, 12.5), sharex=True, sharey=True, dpi=680, facecolor="#ffffff"
)
fig.patch.set_facecolor("#ffffff")
axes_flat = axes.flatten()

for idx, (zone_code, frame_header) in enumerate(zone_config):
  ax = axes_flat[idx]
  apply_journal_axis_style(ax, grid_axis="both", frame_title=frame_header)

  for scenario in ["SSP245", "SSP585"]:
    sub = annual_mme[
        (annual_mme["Scenario"] == scenario)
        & (annual_mme["AGRO_ZONE"] == zone_code)
    ]
    if sub.empty:
      continue
    ax.plot(
        sub["Year"],
        sub["dTmean"],
        label=f"{scenario} MME Mean",
        color=SSP_COLORS[scenario],
        linewidth=4.5,
        zorder=4,
    )
    ax.fill_between(
        sub["Year"],
        sub["dTmean_min"],
        sub["dTmean_max"],
        color=SSP_COLORS[scenario],
        alpha=0.20,
        zorder=2,
    )

  if idx in (0, 2):
    ax.set_ylabel(
        "Temperature Anomaly ΔTmean (°C)",
        fontsize=14,
        fontweight="bold",
        color="#000000",
        labelpad=10,
    )
  if idx in (2, 3):
    ax.set_xlabel(
        "Projection Year (2026–2100)",
        fontsize=14,
        fontweight="bold",
        color="#000000",
        labelpad=10,
    )
  if idx == 0:
    style_bold_legend(ax, loc="upper left")

fig.subplots_adjust(hspace=0.26, wspace=0.14)
export_figure(
    fig,
    os.path.join(
        FIG_OUT_DIR, "Fig11_1_Temperature_Anomaly_Time_Series_2026_2100"
    ),
)

# ------------------------------------------------------------------------------
# STEP 4: Figure 11.2 — Period-Wise Mean Warming Comparison
# ------------------------------------------------------------------------------
print("\n[STEP 4] Generating Figure 11.2 (Period-Wise Warming Comparison)...")

fig, ax = plt.subplots(figsize=(14, 7.5), dpi=680, facecolor="#ffffff")
fig.patch.set_facecolor("#ffffff")
apply_journal_axis_style(ax, grid_axis="y")

period_palette = ["#1F618D", "#283747", "#B9770E"]
sns.barplot(
    data=df_period,
    x="AGRO_ZONE",
    y="dTmean",
    hue="Period",
    palette=period_palette,
    edgecolor="#000000",
    linewidth=3.0,
    ax=ax,
    zorder=3,
)

y_lo, y_hi = ax.get_ylim()
ax.set_ylim(min(0.0, y_lo), y_hi * 1.14)
annotate_bars(ax, fmt="{:+.2f}", offset_ratio=0.02, fontsize=10)

ax.set_ylabel(
    "Mean Warming Anomaly ΔTmean (°C)",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
ax.set_xlabel(
    "Agro-Ecological Zone",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
style_bold_legend(ax, loc="upper left")

plt.tight_layout()
export_figure(
    fig, os.path.join(FIG_OUT_DIR, "Fig11_2_Period_Wise_Warming_Comparison")
)

# ------------------------------------------------------------------------------
# STEP 5: Figure 11.3 — Seasonal / Monthly Warming Cycles (2026-2100)
# ------------------------------------------------------------------------------
print("\n[STEP 5] Generating Figure 11.3 (Monthly Seasonal Warming Cycles)...")

monthly_mme = (
    df_mme.groupby(
        ["Scenario", "Period", "AGRO_ZONE", "Month"], as_index=False
    )["dTmean_mme"]
    .mean()
    .sort_values(by=["Scenario", "Period", "Month"])
)

scenarios_config = [
    ("SSP245", "(a) SSP2-4.5 Seasonal Cycle (OVERALL)"),
    ("SSP585", "(b) SSP5-8.5 Seasonal Cycle (OVERALL)"),
]

fig, axes = plt.subplots(
    1, 2, figsize=(16, 7.2), sharey=True, dpi=680, facecolor="#ffffff"
)
fig.patch.set_facecolor("#ffffff")
cycle_palette = ["#005B96", "#2E8B57", "#C0392B"]
month_labels = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

for idx, (scenario, frame_header) in enumerate(scenarios_config):
  ax = axes[idx]
  apply_journal_axis_style(ax, grid_axis="both", frame_title=frame_header)

  sub = monthly_mme[
      (monthly_mme["Scenario"] == scenario)
      & (monthly_mme["AGRO_ZONE"] == "OVERALL")
  ]
  sns.lineplot(
      data=sub,
      x="Month",
      y="dTmean_mme",
      hue="Period",
      marker="o",
      markersize=10,
      markeredgecolor="#000000",
      markeredgewidth=2.0,
      linewidth=4.5,
      palette=cycle_palette,
      ax=ax,
      zorder=4,
  )

  ax.set_xticks(range(1, 13))
  ax.set_xticklabels(
      month_labels, fontsize=12, fontweight="bold", color="#000000"
  )
  ax.set_xlabel(
      "Calendar Month",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=10,
  )
  if idx == 0:
    ax.set_ylabel(
        "Warming Anomaly ΔTmean (°C)",
        fontsize=14,
        fontweight="bold",
        color="#000000",
        labelpad=10,
    )
  style_bold_legend(ax, loc="upper left")

fig.subplots_adjust(wspace=0.14)
export_figure(
    fig, os.path.join(FIG_OUT_DIR, "Fig11_3_Monthly_Seasonal_Warming_Cycles")
)

# ------------------------------------------------------------------------------
# STEP 6: Figure 11.4 — Nighttime vs. Daytime Asymmetric Warming (OVERALL)
# ------------------------------------------------------------------------------
print(
    "\n[STEP 6] Generating Figure 11.4 (Nighttime vs. Daytime Asymmetric Warming)..."
)

fig, ax = plt.subplots(figsize=(13, 7.5), dpi=680, facecolor="#ffffff")
fig.patch.set_facecolor("#ffffff")
apply_journal_axis_style(ax, grid_axis="y")

df_period_overall = df_period[df_period["AGRO_ZONE"] == "OVERALL"].copy()
period_clean_map = {
    "NEAR_TERM_2026_2050": "2026–2050",
    "MID_TERM_2051_2075": "2051–2075",
    "LONG_TERM_2076_2100": "2076–2100",
}
df_period_overall["Clean_Period"] = df_period_overall["Period"].map(
    lambda x: period_clean_map.get(x, x)
)

x_vals = np.arange(len(df_period_overall))
bar_w = 0.36

bars_tmin = ax.bar(
    x_vals - bar_w / 2.0,
    df_period_overall["dTmin"],
    width=bar_w,
    label="Nighttime Warming (ΔTmin)",
    color="#004C6D",
    edgecolor="#000000",
    linewidth=3.0,
    zorder=3,
)
bars_tmax = ax.bar(
    x_vals + bar_w / 2.0,
    df_period_overall["dTmax"],
    width=bar_w,
    label="Daytime Warming (ΔTmax)",
    color="#C0392B",
    edgecolor="#000000",
    linewidth=3.0,
    zorder=3,
)

max_h = max(
    float(df_period_overall["dTmin"].max()),
    float(df_period_overall["dTmax"].max()),
)
ax.set_ylim(0.0, max_h * 1.16)

for bar_group in (bars_tmin, bars_tmax):
  for b in bar_group:
    h = b.get_height()
    ax.text(
        b.get_x() + b.get_width() / 2.0,
        h + max_h * 0.02,
        f"{h:+.2f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="#000000",
        zorder=5,
    )

clean_tick_labels = [
    f"{r.Scenario}\n({r.Clean_Period})" for _, r in df_period_overall.iterrows()
]
ax.set_xticks(x_vals)
ax.set_xticklabels(
    clean_tick_labels, fontsize=12, fontweight="bold", color="#000000"
)
ax.set_ylabel(
    "Projected Warming (°C)",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
ax.set_xlabel(
    "Scenario & Future Projection Period",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
style_bold_legend(ax, loc="upper left")

plt.tight_layout()
export_figure(
    fig,
    os.path.join(
        FIG_OUT_DIR, "Fig11_4_Daytime_vs_Nighttime_Asymmetric_Warming"
    ),
)

# ------------------------------------------------------------------------------
# STEP 7: Figure 11.5 — Diurnal Temperature Range (DTR) Shift Distributions
# ------------------------------------------------------------------------------
print(
    "\n[STEP 7] Generating Figure 11.5 (Diurnal Temperature Range Shift Boxplots)..."
)

fig, ax = plt.subplots(figsize=(13, 7.5), dpi=680, facecolor="#ffffff")
fig.patch.set_facecolor("#ffffff")
apply_journal_axis_style(ax, grid_axis="y")

dtr_palette = {"SSP245": "#1E7145", "SSP585": "#6A1B9A"}
sns.boxplot(
    data=df_mme,
    x="AGRO_ZONE",
    y="dDTR_mme",
    hue="Scenario",
    palette=dtr_palette,
    linewidth=3.0,
    ax=ax,
    zorder=3,
)

for patch in ax.artists:
  patch.set_edgecolor("#000000")
  patch.set_linewidth(3.0)
for line in ax.lines:
  line.set_color("#000000")
  line.set_linewidth(2.5)

ax.axhline(0.0, color="#000000", linestyle="--", linewidth=4.5, zorder=4)
ax.set_ylabel(
    "DTR Shift ΔDTR (°C)",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
ax.set_xlabel(
    "Agro-Ecological Zone",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
style_bold_legend(ax, loc="best")

plt.tight_layout()
export_figure(
    fig, os.path.join(FIG_OUT_DIR, "Fig11_5_Diurnal_Temperature_Range_Shift")
)

# ------------------------------------------------------------------------------
# STEP 8: Figure 11.6 — Linear Decadal Warming Rates Across Agro-Zones
# ------------------------------------------------------------------------------
print(
    "\n[STEP 8] Generating Figure 11.6 (Agro-Zone Decadal Warming Rates)..."
)

fig, ax = plt.subplots(figsize=(13, 7.5), dpi=680, facecolor="#ffffff")
fig.patch.set_facecolor("#ffffff")
apply_journal_axis_style(ax, grid_axis="y")

rate_palette = {"SSP245": "#1F4E78", "SSP585": "#800000"}
sns.barplot(
    data=df_trends,
    x="AGRO_ZONE",
    y="Rate_dTmean_per_decade",
    hue="Scenario",
    palette=rate_palette,
    edgecolor="#000000",
    linewidth=3.0,
    ax=ax,
    zorder=3,
)

r_lo, r_hi = ax.get_ylim()
ax.set_ylim(min(0.0, r_lo), r_hi * 1.15)
annotate_bars(ax, fmt="{:+.2f}", offset_ratio=0.02, fontsize=11)

ax.set_ylabel(
    "Warming Rate (°C / Decade)",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
ax.set_xlabel(
    "Agro-Ecological Zone",
    fontsize=14,
    fontweight="bold",
    color="#000000",
    labelpad=10,
)
style_bold_legend(ax, loc="upper left")

plt.tight_layout()
export_figure(
    fig, os.path.join(FIG_OUT_DIR, "Fig11_6_AgroZone_Decadal_Warming_Rates")
)

print("\n" + "=" * 88)
print("[COMPLETE] Future projection figures exported with frame labels above frames.")
print(f"Output Directory: {FIG_OUT_DIR}")
print("=" * 88)

## 12. Non-Parametric Mann–Kendall Trend Testing & Probability Density Shifts (2026–2100)

### 12.1 Mann–Kendall & Theil–Sen Trend Analysis
Using annual mean Core-3 Multi-Model Ensemble (`MME`) mean temperature anomalies (`dTmean_mme`, `2026–2100`) per `Scenario` and `AGRO_ZONE`, we apply the non-parametric **Mann–Kendall** test to evaluate monotonic warming trends (`MK_Z_Score`, two-sided `p_value`, and statistical significance at $\alpha = 0.05$) alongside the **Theil–Sen** median pairwise slope scaled to decadal warming rates (`Sen_Slope_C_per_decade`, $^\circ\text{C}/\text{decade}$).

### 12.2 Period-Wise Probability Density Shifts (KDE)
To visualize how projected warming shifts the distribution of monthly mean temperature anomalies (`dTmean_mme`) across `OVERALL` Eritrea from near-term (`2026–2050`) to mid-term (`2051–2075`) and long-term (`2076–2100`) horizons under `SSP245` and `SSP585`, we generate two-panel Kernel Density Estimation (`KDE`) plots with frame labels positioned cleanly above each plot frame.

In [ ]:
# ==============================================================================
# SCRIPT 12.0: MANN-KENDALL TREND SIGNIFICANCE & KDE DENSITY SHIFT PLOTS
# Input Dataset : Core3_MME_Monthly_Anomalies_2026_2100.csv
# Output Table  : Core3_Mann_Kendall_Warming_Trends.csv
# Output Figure : Fig12_1_Temperature_Probability_Density_Shifts.png/.tif/.pdf
# ==============================================================================

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths, Verify Input & Enforce Strict Journal Styling Rules
# ------------------------------------------------------------------------------
BASE_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/1C_TOP_PERFORMING_MODEL_RAW"
)
FIG_OUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/2_FIGURES"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

MME_PATH = os.path.join(BASE_DIR, "Core3_MME_Monthly_Anomalies_2026_2100.csv")
MK_OUT_CSV = os.path.join(BASE_DIR, "Core3_Mann_Kendall_Warming_Trends.csv")

if not os.path.exists(MME_PATH):
  raise FileNotFoundError(
      f"[ERROR] Required MME anomaly file not found at: {MME_PATH}\n"
      "Run Script 9.0 first to generate Core3_MME_Monthly_Anomalies_2026_2100.csv."
  )

# Strict global publication typography and canvas contrast defaults
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

print("=" * 88)
print("SCRIPT 12.0: MANN-KENDALL TREND ANALYSIS & KDE DENSITY SHIFTS (2026-2100)")
print("=" * 88)
print(f"[STEP 1] Input MME Table  : {MME_PATH}")
print(f"         MK Output Table  : {MK_OUT_CSV}")
print(f"         Figure Output    : {FIG_OUT_DIR}")
print("=" * 88)

df_mme = pd.read_csv(MME_PATH)
df_mme.columns = df_mme.columns.str.strip()
df_mme["Date"] = df_mme["Date"].astype(str).str.strip()
df_mme["Year"] = df_mme["Date"].apply(lambda x: int(x.split("-")[0]))
df_mme["dTmean_mme"] = pd.to_numeric(df_mme["dTmean_mme"], errors="coerce")


# ------------------------------------------------------------------------------
# STEP 2: Define Mann-Kendall & Theil-Sen Trend Functions
# ------------------------------------------------------------------------------
def mann_kendall_test(
    series: np.ndarray, alpha: float = 0.05
) -> tuple[float, float, bool]:
  """Computes Mann-Kendall Z-score, two-tailed p-value, and significance flag."""
  vals = np.asarray(series, dtype=float)
  vals = vals[~np.isnan(vals)]
  n = len(vals)
  if n < 3:
    return np.nan, np.nan, False

  s_stat = 0
  for k in range(n - 1):
    diffs = vals[k + 1 :] - vals[k]
    s_stat += int(np.sum(np.sign(diffs)))

  var_s = (n * (n - 1) * (2 * n + 5)) / 18.0
  if var_s <= 0:
    return 0.0, 1.0, False

  if s_stat > 0:
    z_score = (s_stat - 1.0) / np.sqrt(var_s)
  elif s_stat < 0:
    z_score = (s_stat + 1.0) / np.sqrt(var_s)
  else:
    z_score = 0.0

  p_val = float(2.0 * (1.0 - stats.norm.cdf(abs(z_score))))
  is_sig = bool(abs(z_score) > stats.norm.ppf(1.0 - alpha / 2.0))
  return float(z_score), p_val, is_sig


def theil_sen_slope_per_decade(years: np.ndarray, values: np.ndarray) -> float:
  """Computes Theil-Sen median slope scaled to °C per decade."""
  x = np.asarray(years, dtype=float)
  y = np.asarray(values, dtype=float)
  valid = ~np.isnan(x) & ~np.isnan(y)
  x, y = x[valid], y[valid]
  n = len(y)
  if n < 2:
    return np.nan

  pairwise_slopes = []
  for i in range(n - 1):
    dx = x[i + 1 :] - x[i]
    dy = y[i + 1 :] - y[i]
    non_zero = dx != 0
    if np.any(non_zero):
      pairwise_slopes.append(dy[non_zero] / dx[non_zero])

  if not pairwise_slopes:
    return np.nan
  all_slopes = np.concatenate(pairwise_slopes)
  return float(np.median(all_slopes) * 10.0)


# ------------------------------------------------------------------------------
# STEP 3: Compute Annual Zone Trends & Export Mann-Kendall Results Table
# ------------------------------------------------------------------------------
print(
    "\n[STEP 3] Computing annual MME averages, Mann-Kendall Z/p statistics, and Theil-Sen slopes..."
)

annual_mme = (
    df_mme.groupby(["Scenario", "AGRO_ZONE", "Year"], as_index=False)[
        "dTmean_mme"
    ]
    .mean()
    .sort_values(by=["Scenario", "AGRO_ZONE", "Year"])
    .reset_index(drop=True)
)

mk_records = []
for (scenario, zone), grp in annual_mme.groupby(
    ["Scenario", "AGRO_ZONE"], sort=False
):
  z_val, p_val, sig_flag = mann_kendall_test(
      grp["dTmean_mme"].to_numpy(), alpha=0.05
  )
  sen_decade = theil_sen_slope_per_decade(
      grp["Year"].to_numpy(), grp["dTmean_mme"].to_numpy()
  )

  mk_records.append({
      "Scenario": scenario,
      "AGRO_ZONE": zone,
      "MK_Z_Score": round(z_val, 3) if pd.notna(z_val) else np.nan,
      "p_value": round(p_val, 4) if pd.notna(p_val) else np.nan,
      "Significant_p05": sig_flag,
      "Sen_Slope_C_per_decade": (
          round(sen_decade, 3) if pd.notna(sen_decade) else np.nan
      ),
  })

df_mk = pd.DataFrame(mk_records)
df_mk.to_csv(MK_OUT_CSV, index=False)

print("\n" + "=" * 88)
print("TABLE 12.1: MANN-KENDALL & THEIL-SEN WARMING TREND SUMMARY (2026-2100)")
print("=" * 88)
print(df_mk.to_string(index=False))
print("=" * 88)
print(f"  [OK] Exported Mann-Kendall table to:\n       -> {MK_OUT_CSV}")


# ------------------------------------------------------------------------------
# STEP 4: Figure 12.1 — Kernel Density Estimation (KDE) Probability Shifts
# ------------------------------------------------------------------------------
def apply_kde_axis_style(ax: plt.Axes, frame_header: str) -> None:
  """Applies heavy borders, ticks, dashed grid, and frame label ABOVE frame."""
  ax.set_facecolor("#ffffff")
  ax.set_axisbelow(True)

  for side in ["top", "bottom", "left", "right"]:
    spine = ax.spines[side]
    spine.set_visible(True)
    spine.set_color("#000000")
    spine.set_linewidth(3.5)

  ax.tick_params(
      axis="both",
      which="major",
      direction="out",
      length=7,
      width=3.5,
      colors="#000000",
      labelsize=12,
  )
  for lbl in ax.get_xticklabels() + ax.get_yticklabels():
    lbl.set_fontweight("bold")
    lbl.set_color("#000000")
    lbl.set_fontsize(12)

  ax.grid(
      axis="both",
      which="major",
      linestyle="--",
      linewidth=1.5,
      color="#cccccc",
      zorder=0,
  )

  # Frame label positioned cleanly ABOVE the top border
  ax.set_title(
      frame_header,
      loc="left",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      pad=12,
  )


print(
    "\n[STEP 4] Generating Figure 12.1 (KDE Probability Density Shifts across periods)..."
)

period_map = {
    "NEAR_TERM_2026_2050": "2026–2050",
    "MID_TERM_2051_2075": "2051–2075",
    "LONG_TERM_2076_2100": "2076–2100",
}
df_mme["Clean_Period"] = df_mme["Period"].map(lambda x: period_map.get(x, x))

fig, axes = plt.subplots(
    1, 2, figsize=(16, 7.2), sharey=True, dpi=680, facecolor="#ffffff"
)
fig.patch.set_facecolor("#ffffff")

scenarios_config = [
    ("SSP245", "(a) SSP2-4.5 Density Shift (OVERALL)"),
    ("SSP585", "(b) SSP5-8.5 Density Shift (OVERALL)"),
]
kde_colors = {
    "2026–2050": "#005B96",
    "2051–2075": "#E67E22",
    "2076–2100": "#C0392B",
}
period_order = ["2026–2050", "2051–2075", "2076–2100"]

for idx, (scenario_code, frame_label) in enumerate(scenarios_config):
  ax = axes[idx]
  apply_kde_axis_style(ax, frame_label)

  sub = df_mme[
      (df_mme["Scenario"] == scenario_code)
      & (df_mme["AGRO_ZONE"] == "OVERALL")
  ]

  for p_label in period_order:
    p_vals = sub.loc[sub["Clean_Period"] == p_label, "dTmean_mme"].dropna()
    if p_vals.empty:
      continue
    sns.kdeplot(
        p_vals,
        ax=ax,
        label=p_label,
        color=kde_colors[p_label],
        linewidth=4.5,
        fill=True,
        alpha=0.16,
        zorder=4,
    )

  ax.set_xlabel(
      "Temperature Anomaly ΔTmean (°C)",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=10,
  )
  if idx == 0:
    ax.set_ylabel(
        "Probability Density",
        fontsize=14,
        fontweight="bold",
        color="#000000",
        labelpad=10,
    )

  leg = ax.legend(
      loc="upper right",
      frameon=True,
      facecolor="#ffffff",
      edgecolor="#000000",
      fontsize=12,
  )
  if leg is not None:
    leg.get_frame().set_linewidth(3.5)
    for txt in leg.get_texts():
      txt.set_fontweight("bold")
      txt.set_color("#000000")
      txt.set_fontsize(12)

fig.subplots_adjust(wspace=0.14)

out_prefix = os.path.join(
    FIG_OUT_DIR, "Fig12_1_Temperature_Probability_Density_Shifts"
)
fig.savefig(
    f"{out_prefix}.png",
    dpi=680,
    facecolor="#ffffff",
    edgecolor="#000000",
    bbox_inches="tight",
)
fig.savefig(
    f"{out_prefix}.tif",
    dpi=680,
    facecolor="#ffffff",
    edgecolor="#000000",
    bbox_inches="tight",
)
fig.savefig(
    f"{out_prefix}.pdf",
    facecolor="#ffffff",
    edgecolor="#000000",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print(f"  [OK] Exported Figure 12.1 (PNG, TIF at 680 DPI & PDF) to:\n       -> {FIG_OUT_DIR}")
print("\n" + "=" * 88)
print("[COMPLETE] Mann-Kendall trend analysis and KDE probability shift plots finished.")
print("=" * 88)

## 13. Historical 32-Panel Spatial Temperature Maps (ERA5 + 31 CMIP6 Models)

### 13.1 Spatial Comparison Scope
Using historical baseline GeoTIFF rasters (`1981–2014`) across Eritrea and the agro-ecological zone vector boundary (`SELECTED_ZONES_WGS1984.shp`), this script generates three 32-panel spatial grids (arranged on a 6×6 layout with 4 unused bottom-right slots removed):
* **Figure 13.1 (`Tmin`):** Historical mean minimum temperature spatial patterns (`Spectral_r`).
* **Figure 13.2 (`Tmean`):** Historical mean temperature spatial patterns (`YlOrRd`).
* **Figure 13.3 (`Tmax`):** Historical mean maximum temperature spatial patterns (`magma`).

### 13.2 Cartographic & Publication Controls
Across all 32 panels per variable, a unified 2nd–98th percentile color stretch (`vmin` to `vmax`) preserves cross-model comparability, `ERA5_Reanalysis` is highlighted with a distinct crimson border (`#D7191C`), vector boundaries are overlaid cleanly, a North arrow is placed inside each panel, and figures are exported at `680 DPI` in `.png`, `.tif`, and `.pdf` formats.

In [ ]:
# Install required geospatial libraries for reading GeoTIFFs and Shapefiles
!pip install --quiet rioxarray geopandas rasterio

In [ ]:
# ==============================================================================
# SCRIPT 13.0: HISTORICAL 32-PANEL SPATIAL TEMPERATURE MAPS (ERA5 + 31 CMIP6)
# Fix Applied    : Imported xarray as xr and updated `da: xr.DataArray` type hint
#                  (rioxarray extends xarray DataArrays rather than defining DataArray).
# ==============================================================================

import glob
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rxr
import xarray as xr

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths, Load Vector Boundary & Enforce Journal Styling Rules
# ------------------------------------------------------------------------------
TIF_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/Historical_GeoTIFFs"
)
SHP_PATH = (
    "/content/eritrea-cmip6-temperature-evaluation/3_SHAPEFILE_OF_STUDY_AREA/SELECTED_ZONES_WGS1984.shp"
)
FIG_OUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/2_FIGURES"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

print("=" * 88)
print("SCRIPT 13.0: HISTORICAL 32-PANEL SPATIAL TEMPERATURE MAPS (1981-2014)")
print("=" * 88)
print(f"[STEP 1] GeoTIFF Directory : {TIF_DIR}")
print(f"         Shapefile Path    : {SHP_PATH}")
print(f"         Output Directory  : {FIG_OUT_DIR}")
print("=" * 88)

if os.path.exists(SHP_PATH):
  gdf_zone = gpd.read_file(SHP_PATH)
  print(f"  [OK] Loaded study area shapefile ({len(gdf_zone)} features).")
else:
  gdf_zone = None
  print(
      f"  [WARNING] Shapefile not found at {SHP_PATH}. Rendering without boundary overlay."
  )

all_tif_paths = sorted(glob.glob(os.path.join(TIF_DIR, "*.tif")))
print(f"  [OK] Detected {len(all_tif_paths)} GeoTIFF files in directory.")

# Exact 32 datasets: ERA5 baseline reference first + 31 CMIP6 models
VALID_MODELS = [
    "ERA5_Reanalysis",
    "ACCESS_CM2",
    "ACCESS_ESM1_5",
    "BCC_CSM2_MR",
    "CanESM5",
    "CMCC_CM2_SR5",
    "CMCC_ESM2",
    "CNRM_CM6_1",
    "CNRM_ESM2_1",
    "EC_Earth3",
    "EC_Earth3_Veg_LR",
    "FGOALS_g3",
    "GFDL_CM4",
    "GFDL_ESM4",
    "GISS_E2_1_G",
    "HadGEM3_GC31_LL",
    "HadGEM3_GC31_MM",
    "INM_CM4_8",
    "INM_CM5_0",
    "IPSL_CM6A_LR",
    "KACE_1_0_G",
    "KIOST_ESM",
    "MIROC_ES2L",
    "MIROC6",
    "MPI_ESM1_2_HR",
    "MPI_ESM1_2_LR",
    "MRI_ESM2_0",
    "NESM3",
    "NorESM2_LM",
    "NorESM2_MM",
    "TaiESM1",
    "UKESM1_0_LL",
]

VAR_CONFIG = {
    "Tmin": {
        "cmap": "Spectral_r",
        "fig_prefix": "Fig13_1_Historical_Spatial_Tmin_31Models_ERA5",
    },
    "Tmean": {
        "cmap": "YlOrRd",
        "fig_prefix": "Fig13_2_Historical_Spatial_Tmean_31Models_ERA5",
    },
    "Tmax": {
        "cmap": "magma",
        "fig_prefix": "Fig13_3_Historical_Spatial_Tmax_31Models_ERA5",
    },
}


# ------------------------------------------------------------------------------
# STEP 2: Raster Matching, CRS Alignment & Cartographic Helpers
# ------------------------------------------------------------------------------
def find_tif_file(
    mod_name: str, var_name: str, tif_list: list[str]
) -> str | None:
  """Matches model name and temperature variable against GeoTIFF filenames."""
  mod_key = mod_name.lower().replace("_", "").replace("-", "")
  var_key = var_name.lower()
  for path in tif_list:
    fname = os.path.basename(path).lower().replace("_", "").replace("-", "")
    if mod_key in fname and fname.endswith(f"{var_key}.tif"):
      return path
  return None


def align_vector_to_raster(
    gdf: gpd.GeoDataFrame | None, da: xr.DataArray
) -> gpd.GeoDataFrame | None:
  """Reprojects shapefile to match raster CRS when both are defined."""
  if gdf is None or gdf.empty:
    return None
  try:
    raster_crs = da.rio.crs
    if gdf.crs is not None and raster_crs is not None and gdf.crs != raster_crs:
      return gdf.to_crs(raster_crs)
  except Exception:
    pass
  return gdf


def add_north_arrow(ax: plt.Axes) -> None:
  """Draws a high-contrast North arrow inside the upper-right frame corner."""
  ax.annotate(
      "N",
      xy=(0.90, 0.89),
      xytext=(0.90, 0.73),
      xycoords="axes fraction",
      textcoords="axes fraction",
      arrowprops=dict(
          facecolor="#000000",
          edgecolor="#000000",
          width=2.5,
          headwidth=7.5,
          headlength=7.5,
      ),
      ha="center",
      va="bottom",
      fontsize=10,
      fontweight="bold",
      color="#000000",
      zorder=8,
  )


# ------------------------------------------------------------------------------
# STEP 3: Render 32-Panel Spatial Grids for Tmin, Tmean, and Tmax
# ------------------------------------------------------------------------------
print(
    "\n[STEP 3] Rendering 32-panel spatial maps across Tmin, Tmean, and Tmax..."
)

for var_name, cfg in VAR_CONFIG.items():
  print(f"\n  -> Processing variable: {var_name}...")

  data_store: dict[str, xr.DataArray | None] = {}
  valid_arrays: list[np.ndarray] = []

  for mod_name in VALID_MODELS:
    matched_tif = find_tif_file(mod_name, var_name, all_tif_paths)
    if matched_tif is None:
      print(f"     [WARNING] Missing GeoTIFF for {mod_name} ({var_name}).")
      data_store[mod_name] = None
      continue

    try:
      da = rxr.open_rasterio(matched_tif, masked=True).squeeze()
      data_store[mod_name] = da
      vals = da.to_numpy()
      finite_vals = vals[np.isfinite(vals)]
      if finite_vals.size > 0:
        valid_arrays.append(finite_vals)
    except Exception as exc:
      print(f"     [WARNING] Failed reading {mod_name} ({var_name}): {exc}")
      data_store[mod_name] = None

  if not valid_arrays:
    print(f"     [ERROR] No valid raster values found for {var_name}. Skipping.")
    continue

  # Shared 2nd-98th percentile color stretch across all available panels
  pooled_vals = np.concatenate(valid_arrays)
  vmin = float(np.percentile(pooled_vals, 2))
  vmax = float(np.percentile(pooled_vals, 98))

  fig, axes = plt.subplots(
      6,
      6,
      figsize=(24, 24),
      dpi=680,
      facecolor="#ffffff",
      subplot_kw={"aspect": "equal"},
  )
  fig.patch.set_facecolor("#ffffff")
  axes_flat = axes.flatten()

  im_mappable = None
  for idx, mod_name in enumerate(VALID_MODELS):
    ax = axes_flat[idx]
    ax.set_facecolor("#ffffff")
    da = data_store.get(mod_name)

    if da is not None:
      im_mappable = da.plot(
          ax=ax,
          cmap=cfg["cmap"],
          vmin=vmin,
          vmax=vmax,
          add_colorbar=False,
      )
      gdf_aligned = align_vector_to_raster(gdf_zone, da)
      if gdf_aligned is not None:
        gdf_aligned.plot(
            ax=ax, facecolor="none", edgecolor="#000000", linewidth=1.5
        )

    # Frame label cleanly ABOVE the top border
    ax.set_title(
        mod_name,
        loc="left",
        fontsize=12,
        fontweight="bold",
        color="#000000",
        pad=8,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Harden all 4 spines and ticks (ERA5 highlighted in crimson red)
    is_ref = mod_name == "ERA5_Reanalysis"
    spine_color = "#D7191C" if is_ref else "#000000"
    spine_width = 4.0 if is_ref else 3.5

    for side in ["top", "bottom", "left", "right"]:
      sp = ax.spines[side]
      sp.set_visible(True)
      sp.set_color(spine_color)
      sp.set_linewidth(spine_width)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=5,
        width=2.5,
        colors="#000000",
        labelsize=9,
    )
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
      lbl.set_fontweight("bold")
      lbl.set_color("#000000")
      lbl.set_fontsize(9)

    add_north_arrow(ax)

  # Remove unused bottom-right subplots (indices 32..35)
  for empty_idx in range(len(VALID_MODELS), len(axes_flat)):
    fig.delaxes(axes_flat[empty_idx])

  # Shared horizontal colorbar with hardened frame and bold 14pt label
  cbar_ax = fig.add_axes([0.28, 0.04, 0.44, 0.016])
  cbar = fig.colorbar(im_mappable, cax=cbar_ax, orientation="horizontal")
  cbar.outline.set_edgecolor("#000000")
  cbar.outline.set_linewidth(3.5)
  cbar.ax.tick_params(length=7, width=3.5, colors="#000000", labelsize=12)
  for lbl in cbar.ax.get_xticklabels():
    lbl.set_fontweight("bold")
    lbl.set_color("#000000")
    lbl.set_fontsize(12)
  cbar.set_label(
      f"1981–2014 Baseline Mean {var_name} (°C)",
      fontsize=14,
      fontweight="bold",
      color="#000000",
      labelpad=10,
  )

  plt.subplots_adjust(
      left=0.04, right=0.96, top=0.95, bottom=0.09, wspace=0.28, hspace=0.40
  )

  out_prefix = os.path.join(FIG_OUT_DIR, cfg["fig_prefix"])
  out_png = f"{out_prefix}.png"
  out_tif = f"{out_prefix}.tif"
  out_pdf = f"{out_prefix}.pdf"

  fig.savefig(
      out_png,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      out_tif,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      out_pdf,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  plt.show()
  plt.close(fig)

  print(
      f"     [OK] Exported 680-DPI spatial grid:\n          -> {out_png}\n          -> {out_tif}"
  )

print("\n" + "=" * 88)
print("[COMPLETE] All three 32-panel historical spatial grids generated.")
print("=" * 88)

## 14. Multi-Scenario Spatial Heterogeneity & Anomaly Maps (1985–2014 vs. 2026–2100)

### 14.1 Five-Panel Spatial Structure
Using the Core-3 Multi-Model Ensemble (`MME`) GeoTIFF rasters and study area vector boundary (`SELECTED_ZONES_WGS1984.shp`), this script generates a five-panel (`1 × 5`) spatial comparison figure for each temperature variable (`Tmin`, `Tmean`, and `Tmax`):
* **Panel (a):** Historical MME baseline (`1985–2014`).
* **Panel (b):** Projected `SSP245` MME mean (`2026–2100`).
* **Panel (c):** Projected `ΔSSP245` anomaly relative to the `1985–2014` baseline (`coolwarm`, crimson border).
* **Panel (d):** Projected `SSP585` MME mean (`2026–2100`).
* **Panel (e):** Projected `ΔSSP585` anomaly relative to the `1985–2014` baseline (`coolwarm`, crimson border).

### 14.2 Cartographic & Publication Standards
All figures are exported at `680 DPI` (`.png` and `.tif` plus vector `.pdf`) on a pure white canvas with bold black typography, hardened spines (`3.5` black for standard panels; `4.0` crimson for anomaly panels), thick ticks, a North arrow inside each frame, and dedicated bottom colorbars per panel.

In [ ]:
# ==============================================================================
# SCRIPT 14.0: MULTI-SCENARIO SPATIAL HETEROGENEITY & ANOMALY MAPS
# Fix Applied    : Imported xarray as xr and updated `da: xr.DataArray` type hint.
# ==============================================================================

import os
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rxr
import xarray as xr

# ------------------------------------------------------------------------------
# STEP 1: Configure Paths, Load Shapefile & Enforce Strict Journal Styling Rules
# ------------------------------------------------------------------------------
DATA_DIR = (
    "/content/eritrea-cmip6-temperature-evaluation/1_DATA/Future_GeoTIFFs"
)
SHP_PATH = (
    "/content/eritrea-cmip6-temperature-evaluation/3_SHAPEFILE_OF_STUDY_AREA/SELECTED_ZONES_WGS1984.shp"
)
FIG_OUT_DIR = "/content/eritrea-cmip6-temperature-evaluation/2_FIGURES"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["text.color"] = "#000000"
plt.rcParams["axes.edgecolor"] = "#000000"
plt.rcParams["axes.linewidth"] = 3.5
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#ffffff"

print("=" * 88)
print("SCRIPT 14.0: MULTI-SCENARIO SPATIAL HETEROGENEITY & ANOMALY MAPS")
print("=" * 88)
print(f"[STEP 1] Future Raster Dir : {DATA_DIR}")
print(f"         Shapefile Path    : {SHP_PATH}")
print(f"         Output Directory  : {FIG_OUT_DIR}")
print("=" * 88)

if os.path.exists(SHP_PATH):
  gdf_zone = gpd.read_file(SHP_PATH)
  print(f"  [OK] Loaded study area shapefile ({len(gdf_zone)} features).")
else:
  gdf_zone = None
  print(
      f"  [WARNING] Shapefile not found at {SHP_PATH}. Rendering without vector boundaries."
  )

VAR_CONFIGS = {
    "Tmin": {
        "name": "Minimum Temperature (Tmin)",
        "base_cmap": "Spectral_r",
        "anom_cmap": "coolwarm",
    },
    "Tmean": {
        "name": "Mean Temperature (Tmean)",
        "base_cmap": "YlOrRd",
        "anom_cmap": "coolwarm",
    },
    "Tmax": {
        "name": "Maximum Temperature (Tmax)",
        "base_cmap": "magma",
        "anom_cmap": "coolwarm",
    },
}


# ------------------------------------------------------------------------------
# STEP 2: CRS Alignment & Cartographic North Arrow Helpers
# ------------------------------------------------------------------------------
def align_vector_to_raster(
    gdf: gpd.GeoDataFrame | None, da: xr.DataArray
) -> gpd.GeoDataFrame | None:
  """Reprojects vector boundaries to match the raster CRS when available."""
  if gdf is None or gdf.empty:
    return None
  try:
    raster_crs = da.rio.crs
    if gdf.crs is not None and raster_crs is not None and gdf.crs != raster_crs:
      return gdf.to_crs(raster_crs)
  except Exception:
    pass
  return gdf


def add_north_arrow(ax: plt.Axes) -> None:
  """Draws a high-contrast North arrow inside the upper-right frame corner."""
  ax.annotate(
      "N",
      xy=(0.88, 0.89),
      xytext=(0.88, 0.73),
      xycoords="axes fraction",
      textcoords="axes fraction",
      arrowprops=dict(
          facecolor="#000000",
          edgecolor="#000000",
          width=2.5,
          headwidth=7.5,
          headlength=7.5,
      ),
      ha="center",
      va="bottom",
      fontsize=10,
      fontweight="bold",
      color="#000000",
      zorder=8,
  )


# ------------------------------------------------------------------------------
# STEP 3: Generate 5-Panel Spatial Maps Across Tmin, Tmean, and Tmax
# ------------------------------------------------------------------------------
print(
    "\n[STEP 3] Rendering 5-panel spatial heterogeneity and anomaly figures..."
)

for var_code, cfg in VAR_CONFIGS.items():
  print(f"\n  -> Processing variable: {var_code} ({cfg['name']})...")

  hist_tif = os.path.join(
      DATA_DIR, f"CMIP6_Historical_1985_2014_MME_{var_code}.tif"
  )
  ssp245_tif = os.path.join(
      DATA_DIR, f"CMIP6_SSP245_2026_2100_MME_{var_code}.tif"
  )
  ssp245_anom_tif = os.path.join(
      DATA_DIR, f"CMIP6_SSP245_Anomaly_2026_2100_{var_code}.tif"
  )
  ssp585_tif = os.path.join(
      DATA_DIR, f"CMIP6_SSP585_2026_2100_MME_{var_code}.tif"
  )
  ssp585_anom_tif = os.path.join(
      DATA_DIR, f"CMIP6_SSP585_Anomaly_2026_2100_{var_code}.tif"
  )

  required_files = [
      hist_tif,
      ssp245_tif,
      ssp245_anom_tif,
      ssp585_tif,
      ssp585_anom_tif,
  ]
  missing_files = [p for p in required_files if not os.path.exists(p)]
  if missing_files:
    print(
        f"     [WARNING] Missing raster file(s) for {var_code}: {missing_files}. Skipping."
    )
    continue

  try:
    da_hist = rxr.open_rasterio(hist_tif, masked=True).squeeze()
    da_ssp245 = rxr.open_rasterio(ssp245_tif, masked=True).squeeze()
    da_ssp245_anom = rxr.open_rasterio(ssp245_anom_tif, masked=True).squeeze()
    da_ssp585 = rxr.open_rasterio(ssp585_tif, masked=True).squeeze()
    da_ssp585_anom = rxr.open_rasterio(ssp585_anom_tif, masked=True).squeeze()
  except Exception as exc:
    print(f"     [ERROR] Failed reading GeoTIFFs for {var_code}: {exc}")
    continue

  fig, axes = plt.subplots(
      1,
      5,
      figsize=(27, 7.5),
      dpi=680,
      facecolor="#ffffff",
      subplot_kw={"aspect": "equal"},
  )
  fig.patch.set_facecolor("#ffffff")

  panels = [
      (
          da_hist,
          "(a) Historical\n(1985–2014)",
          cfg["base_cmap"],
          False,
      ),
      (
          da_ssp245,
          "(b) SSP2-4.5\n(2026–2100)",
          cfg["base_cmap"],
          False,
      ),
      (
          da_ssp245_anom,
          "(c) ΔSSP2-4.5 Anomaly\n(vs. 1985–2014)",
          cfg["anom_cmap"],
          True,
      ),
      (
          da_ssp585,
          "(d) SSP5-8.5\n(2026–2100)",
          cfg["base_cmap"],
          False,
      ),
      (
          da_ssp585_anom,
          "(e) ΔSSP5-8.5 Anomaly\n(vs. 1985–2014)",
          cfg["anom_cmap"],
          True,
      ),
  ]

  for idx, (da, frame_label, cmap_name, is_anomaly) in enumerate(panels):
    ax = axes[idx]
    ax.set_facecolor("#ffffff")

    raw_vals = da.to_numpy()
    finite_vals = raw_vals[np.isfinite(raw_vals)]
    if finite_vals.size > 0:
      if is_anomaly:
        abs_max = float(np.percentile(np.abs(finite_vals), 98))
        abs_max = max(abs_max, 0.1)
        vmin, vmax = -abs_max, abs_max
      else:
        vmin = float(np.percentile(finite_vals, 2))
        vmax = float(np.percentile(finite_vals, 98))
        if vmin == vmax:
          vmax = vmin + 0.1
    else:
      vmin, vmax = 0.0, 1.0

    im = da.plot(
        ax=ax,
        cmap=cmap_name,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )

    gdf_aligned = align_vector_to_raster(gdf_zone, da)
    if gdf_aligned is not None:
      gdf_aligned.plot(
          ax=ax, facecolor="none", edgecolor="#000000", linewidth=1.5
      )

    # Frame label placed cleanly ABOVE the top border
    ax.set_title(
        frame_label,
        loc="left",
        fontsize=13,
        fontweight="bold",
        color="#000000",
        pad=10,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")

    border_color = "#D7191C" if is_anomaly else "#000000"
    border_width = 4.0 if is_anomaly else 3.5
    for side in ["top", "bottom", "left", "right"]:
      sp = ax.spines[side]
      sp.set_visible(True)
      sp.set_color(border_color)
      sp.set_linewidth(border_width)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=6,
        width=3.0,
        colors="#000000",
        labelsize=10,
    )
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
      lbl.set_fontweight("bold")
      lbl.set_color("#000000")
      lbl.set_fontsize(10)

    add_north_arrow(ax)

    cbar = fig.colorbar(
        im, ax=ax, orientation="horizontal", pad=0.14, shrink=0.88
    )
    cbar.outline.set_edgecolor("#000000")
    cbar.outline.set_linewidth(3.0)
    cbar.ax.tick_params(length=6, width=3.0, colors="#000000", labelsize=10)
    for lbl in cbar.ax.get_xticklabels():
      lbl.set_fontweight("bold")
      lbl.set_color("#000000")
      lbl.set_fontsize(10)

    cbar_label = f"Δ {var_code} (°C)" if is_anomaly else f"Mean {var_code} (°C)"
    cbar.set_label(
        cbar_label, fontsize=12, fontweight="bold", color="#000000", labelpad=8
    )

  plt.tight_layout()

  out_prefix = os.path.join(
      FIG_OUT_DIR, f"Fig14_Spatial_Heterogeneity_MultiDecadal_{var_code}"
  )
  out_png = f"{out_prefix}.png"
  out_tif = f"{out_prefix}.tif"
  out_pdf = f"{out_prefix}.pdf"

  fig.savefig(
      out_png,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      out_tif,
      dpi=680,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  fig.savefig(
      out_pdf,
      facecolor="#ffffff",
      edgecolor="#000000",
      bbox_inches="tight",
  )
  plt.show()
  plt.close(fig)

  print(
      f"     [OK] Exported 680-DPI multi-scenario map:\n          -> {out_png}\n          -> {out_tif}"
  )

print("\n" + "=" * 88)
print("[COMPLETE] All multi-scenario spatial anomaly figures generated.")
print("=" * 88)